# RAG retrieval: historical experiments and current train/test/holdout validation

Earlier outputs are retained for provenance. The final section below uses the newly supplied partitions and active snapshot.

In [1]:
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import sys
from pathlib import Path

candidates = [Path(os.environ["AIAGENT_ROOT"])] if "AIAGENT_ROOT" in os.environ else []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "apps/aiAgent", parent / "frappe-bench/apps/aiAgent"])
ROOT = next((p.resolve() for p in candidates if (p / "rag/data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set AIAGENT_ROOT to the absolute aiAgent folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rag.data import config  # noqa: E402 - project root must be resolved first

cfg = config()
print("Workspace:", ROOT)

Workspace: /Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent


In [2]:
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from IPython.utils.io import capture_output

from rag.data import NORMALIZED, digest, load_corpus, read, write_json
from rag.db import connect
from rag.embedding import CrossEncoder, Embedder
from rag.evaluation import (
    CONFIGURATIONS,
    benchmark_split,
    branch_cache,
    discover_gold,
    query_for,
    ranking_metrics,
    sweep_router,
    sweep_weights,
    weight_grid,
)
from rag.retrieval import Retriever, candidate_paths, fuse, rerank_candidates, unique_codes

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
bundle = load_corpus()
conn = connect()
embedder = Embedder(cfg)
retriever = Retriever(conn, embedder)
print("Snapshot:", retriever.snapshot, "Profile:", retriever.profile)
assert retriever.snapshot == bundle["snapshot"]

/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Fetching 22 files: 100%|██████████| 22/22 [00:00<00:00, 3874.65it/s]

Snapshot: 1a67df40d28f8b07a668746c Profile: 3a2ce6e6d726a0f053748717


## Supplied partitions and query audit
Read only `data/verified/hs/train/*.jsonl` and `data/verified/hs/test/*.jsonl`.
Ignore CSV mirrors and `all/`. Keep the user split, report overlap instead of resplitting.
Strip explicit tariff/Chapter labels and dataset instructions; never use ground_truth_condition or answer wording.

In [3]:
gold, gold_files = discover_gold()
assert gold and {r["split"] for r in gold} == {"train", "test"}
assert {r["expected_hs_8"] for r in gold} <= {r.get("code") for r in bundle["records"]}
split = benchmark_split(gold)
print("Sources:", gold_files, "Train/test:", len(split["dev"]), len(split["test"]))
print("Overlapping HS labels (not duplicate products):", split["overlapping_codes"])
print("Gold hash:", digest(gold))
display(pd.DataFrame(gold).groupby(["split", "section"]).agg(
    products=("test_id", "count"), hs_codes=("expected_hs_8", "nunique")))
cases = []
cleaning = []
for row in gold:
    raw = row["product_name_vi"] + ". " + row["technical_description_for_test"]
    clean = query_for(row, "name_and_spec")
    if raw != clean:
        cleaning.append({"id": row["test_id"], "before": raw, "after": clean})
    for variant in ("name_only", "name_and_spec"):
        cases.append({**row, "base_id": row["test_id"],
                      "test_id": row["test_id"] + ":" + variant,
                      "variant": variant, "query": query_for(row, variant)})
dev_cases = [c for c in cases if c["split"] == "train"]
test_cases = [c for c in cases if c["split"] == "test"]
display(pd.DataFrame(cleaning))
print("Query variants:", len(dev_cases), "train;", len(test_cases), "test")

Sources: ['data/verified/hs/train/HS_RAG_TRAIN_300_VERIFIED.jsonl', 'data/verified/hs/test/HS_RAG_TEST_90_DISJOINT_VERIFIED.jsonl'] Train/test: 300 90
Overlapping HS labels (not duplicate products): ['71131190', '71131990', '71132090', '71141100', '71141900', '71142000', '71171920', '71179020']
Gold hash: 2b3fc4bdee12212e9eeb764e270f1dd0e3e721194df94a7b9675b32b820bd6aa


products  hs_codes
split section                    
test  XIV            30         8
      XVI            30        30
      XVII           30        30
train XIV           100        80
      XVI           100        95
      XVII          100       100

,id,before,after
0,XIV_TR001,"Ngọc trai tự nhiên. Ngọc trai tự nhiên, chưa xâu chuỗi, chưa gắn hoặc nạm dát; thuộc nhóm 71.01.","Ngọc trai tự nhiên. Ngọc trai tự nhiên, chưa xâu chuỗi, chưa gắn hoặc nạm dát"
1,XIV_TR002,"Ngọc trai nuôi cấy chưa gia công. Ngọc trai nuôi cấy ở trạng thái chưa gia công, chưa xâu chuỗi/gắn/nạm dát.","Ngọc trai nuôi cấy chưa gia công. Ngọc trai nuôi cấy ở trạng thái chưa gia công, chưa xâu chuỗi/gắn/nạm dát"
2,XIV_TR003,"Ngọc trai nuôi cấy đã gia công. Ngọc trai nuôi cấy đã được gia công nhưng chưa xâu chuỗi, chưa gắn hoặc nạm dát.","Ngọc trai nuôi cấy đã gia công. Ngọc trai nuôi cấy đã được gia công nhưng chưa xâu chuỗi, chưa gắn hoặc nạm dát"
3,XIV_TR004,"Kim cương chưa phân loại. Kim cương tự nhiên chưa được phân loại, chưa gắn hoặc nạm dát.","Kim cương chưa phân loại. Kim cương tự nhiên chưa được phân loại, chưa gắn hoặc nạm dát"
4,XIV_TR005,Kim cương công nghiệp chưa gia công. Kim cương công nghiệp ở dạng chưa gia công hoặc chỉ gia công sơ bộ.,Kim cương công nghiệp chưa gia công. Kim cương công nghiệp ở dạng chưa gia công hoặc chỉ gia công sơ bộ
5,XIV_TR006,Kim cương công nghiệp loại khác. Kim cương công nghiệp đã gia công theo cách khác so với dòng 7102.21.,Kim cương công nghiệp loại khác. Kim cương công nghiệp đã gia công theo cách khác so với dòng 7102.21
6,XIV_TR007,Kim cương phi công nghiệp chưa gia công. Kim cương phi công nghiệp chưa gia công hoặc chỉ gia công sơ bộ; chưa gắn/nạm dát.,Kim cương phi công nghiệp chưa gia công. Kim cương phi công nghiệp chưa gia công hoặc chỉ gia công sơ bộ; chưa gắn/nạm dát
7,XIV_TR008,Kim cương phi công nghiệp loại khác. Kim cương phi công nghiệp đã gia công theo cách khác so với 7102.31; chưa gắn/nạm dát.,Kim cương phi công nghiệp loại khác. Kim cương phi công nghiệp đã gia công theo cách khác so với 7102.31; chưa gắn/nạm dát
8,XIV_TR009,Rubi thô. Đá rubi tự nhiên chưa gia công hoặc chỉ cắt đơn giản/tạo hình thô; chưa gắn/nạm dát.,Rubi thô. Đá rubi tự nhiên chưa gia công hoặc chỉ cắt đơn giản/tạo hình thô; chưa gắn/nạm dát
9,XIV_TR010,Ngọc bích thô. Ngọc bích nephrite hoặc jadeite tự nhiên ở trạng thái thô/đơn giản.,Ngọc bích thô. Ngọc bích nephrite hoặc jadeite tự nhiên ở trạng thái thô/đơn giản


Query variants: 600 train; 180 test


## 2. Metadata, graph, template registry và legal source qrels
Kiểm tra filter nhiều tầng và namespaces trực tiếp trên database. GIR thuộc hs_classification; TT85 thuộc legal_classification. 5 mẫu chỉ kiểm thử tra cứu template, không tính vào legal retrieval recall.

In [4]:
metadata_rows = conn.execute(
    "SELECT namespace,dataset,chunk_type,section,level,count(*) FROM rag.chunks WHERE snapshot=%s GROUP BY 1,2,3,4,5 ORDER BY 1,2,4,5",
    (retriever.snapshot,),
).fetchall()
display(pd.DataFrame(metadata_rows))
for level in (2, 4, 6, 8):
    found = retriever.documents("knowledge", namespace="hs_classification", levels=[level])
    assert found and all(r["level"] == level for r in found)
assert all(
    r["namespace"] == "legal_classification" and r["chunk_type"] == "legal_clause"
    for r in retriever.documents("tt85")
)
assert not any("TT85-FORM" in r["id"] for r in bundle["records"])
forms = read(NORMALIZED / "TT85/tt85_forms_source.json")
template_tests = [
    {
        "id": "T" + f["form_no"],
        "code": f["code"],
        "passed": next(x for x in forms if x["code"] == f["code"])["title"] == f["title"],
    }
    for f in forms
]
display(pd.DataFrame(template_tests))
assert all(r["passed"] for r in template_tests)
print("Namespace, hierarchy filters, form exclusion: PASS")

,namespace,dataset,chunk_type,section,level,count
0,hs_classification,gir,gir_explanation,None,NaN,104
1,hs_classification,gir,gir_rule,None,NaN,11
2,hs_classification,scope,None,I,2.0,5
3,hs_classification,scope,None,II,2.0,9
4,hs_classification,scope,None,III,2.0,1
5,hs_classification,scope,None,IV,2.0,9
6,hs_classification,scope,None,IX,2.0,3
7,hs_classification,scope,None,V,2.0,3
8,hs_classification,scope,None,VI,2.0,11
9,hs_classification,scope,None,VII,2.0,2


,id,code,passed
0,T01,01/PYCPT/2026,True
1,T02,02/PTNYCPT/2026,True
2,T03,03/BBTLMHH/2026,True
3,T04,04/TBKQPTPL/2026,True
4,T05,05/PGTL/2026,True


Namespace, hierarchy filters, form exclusion: PASS


In [5]:

from rag.data import ROOT, load_corpus, read
from rag.evaluation import discover_qa

qa_cases = discover_qa(load_corpus())
assert len(qa_cases) == 650
print('Supplied QA partitions:', Counter(c['split'] for c in qa_cases))
print('All exact export aliases resolve to current corpus chunks. No gold is ingested.')
previous_selection = read(ROOT / 'config/selected_retrieval.json')
legal_policy = previous_selection['legal']

Supplied QA partitions: Counter({'train': 320, 'test': 150})
All exact export aliases resolve to current corpus chunks. No gold is ingested.


## C1–C15 weight and RRF search on train
C1–C10 original families; C11 full hybrid + cross-encoder; C12 dense/sparse + ColBERT.
Additional families: C13 dense/sparse/BM25/Exact, C14 dense/sparse/BM25, C15 sparse/BM25/Exact.
Search positive .05 simplex weights plus 600 deterministic Dirichlet samples per hybrid family.
RRF k = 5, 10, 20, 40, 60, 100, 200. Queries retrieve each branch once; cached ranks support repeatable sweeps.

In [6]:
KS = (5, 10, 20, 40, 60, 100, 200)
grid_count = sum(len(weight_grid(branches)) * len(KS) for branches in CONFIGURATIONS.values())
print("Weight/k trials per Section:", grid_count)
with capture_output() as branch_progress:
    dev_cache = branch_cache(retriever, dev_cases, filters={"levels": [8]}, pool=100)
    dev_router_cache = branch_cache(retriever, dev_cases,
        filters={"sections": ["XIV", "XVI", "XVII"]}, pool=100)
print("Cached TRAIN leaf/router:", len(dev_cache), len(dev_router_cache))


Weight/k trials per Section: 78253


Cached TRAIN leaf/router: 600 600


In [7]:
def objective(row):
    return row["hit@1"], row["ahtn8_recall@5"], row["mrr"], row["ndcg@10"]

section_sweeps = {}
section_policies = {}
summary = []
for section in ("XIV", "XVI", "XVII"):
    subset = [item for item in dev_cache if item["case"]["section"] == section]
    section_sweeps[section] = {}
    for family in CONFIGURATIONS:
        runs = sweep_weights(subset, family, KS)
        section_sweeps[section][family] = runs[:10]
        summary.append({"section": section, **runs[0], "trials": len(runs)})
    chosen = max([r for r in summary if r["section"] == section], key=objective)
    section_policies[section] = {k: chosen[k] for k in ("configuration", "weights", "k")}
    print("TRAIN selected base", section, section_policies[section], flush=True)
display(pd.DataFrame(summary))
router_sweep = sweep_router(dev_router_cache)
router_policy = router_sweep[0]
display(pd.DataFrame(router_sweep[:10]))
print("Router TRAIN only:", router_policy)

TRAIN selected base XIV {'configuration': 'C9', 'weights': {'dense': 0.35, 'bm25': 0.35, 'exact': 0.3}, 'k': 20}


TRAIN selected base XVI {'configuration': 'C13', 'weights': {'dense': 0.2356912907696891, 'sparse': 0.27843193151228, 'bm25': 0.26126743617756276, 'exact': 0.2246093415404681}, 'k': 5}


TRAIN selected base XVII {'configuration': 'C14', 'weights': {'dense': 0.4, 'sparse': 0.1, 'bm25': 0.5}, 'k': 5}


,section,configuration,weights,k,count,hit@1,ahtn8_recall@5,mrr,ndcg@10,trials
0,XIV,C1,{'dense': 1.0},5,200,0.395,0.775,0.546067,0.622006,7
1,XIV,C2,{'sparse': 1.0},5,200,0.400,0.760,0.556768,0.623837,7
2,XIV,C3,{'bm25': 1.0},5,200,0.470,0.790,0.596595,0.662472,7
3,XIV,C4,{'fts': 1.0},5,200,0.105,0.220,0.148698,0.179583,7
4,XIV,C5,"{'dense': 0.4, 'sparse': 0.6}",10,200,0.440,0.805,0.589859,0.658218,4361
5,XIV,C6,"{'dense': 0.4104011295335827, 'bm25': 0.5895988704664173}",10,200,0.495,0.845,0.626982,0.692305,4354
6,XIV,C7,"{'dense': 0.4659134648887171, 'sparse': 0.47803229031316746, 'fts': 0.056054244798115445}",10,200,0.455,0.820,0.595956,0.664024,5432
7,XIV,C8,"{'dense': 0.3545648757459851, 'sparse': 0.5318745933303629, 'exact': 0.113560530923652}",60,200,0.440,0.805,0.591530,0.657475,5432
8,XIV,C9,"{'dense': 0.35, 'bm25': 0.35, 'exact': 0.3}",20,200,0.495,0.845,0.631331,0.699911,5432
9,XIV,C10,"{'dense': 0.2362063930546834, 'sparse': 0.04719840802210506, 'bm25': 0.538615980853112, 'fts': 0.16652548354897692, 'exact': 0.011453734521122722}",5,200,0.490,0.835,0.612895,0.677612,31332


,weights,k,section_accuracy_macro,section_recall@2_macro
0,"{'dense': 0.4, 'sparse': 0.15, 'bm25': 0.35, 'fts': 0.05, 'exact': 0.05}",60,0.956667,0.998333
1,"{'dense': 0.10637156341874121, 'sparse': 0.0010628794779150032, 'bm25': 0.8764617297816047, 'fts': 0.002236896428416138, 'exact': 0.013866930893322969}",100,0.956667,0.998333
2,"{'dense': 0.3, 'sparse': 0.15, 'bm25': 0.45, 'fts': 0.05, 'exact': 0.05}",20,0.953333,0.998333
3,"{'dense': 0.35, 'sparse': 0.15, 'bm25': 0.4, 'fts': 0.05, 'exact': 0.05}",60,0.953333,0.998333
4,"{'dense': 0.2903925556966506, 'sparse': 0.0012777357374981234, 'bm25': 0.6695776583429442, 'fts': 0.005010420525958629, 'exact': 0.033741629696948605}",20,0.953333,0.998333
5,"{'dense': 0.2903925556966506, 'sparse': 0.0012777357374981234, 'bm25': 0.6695776583429442, 'fts': 0.005010420525958629, 'exact': 0.033741629696948605}",100,0.953333,0.998333
6,"{'dense': 0.25, 'sparse': 0.05, 'bm25': 0.6, 'fts': 0.05, 'exact': 0.05}",20,0.951667,0.998333
7,"{'dense': 0.35, 'sparse': 0.2, 'bm25': 0.35, 'fts': 0.05, 'exact': 0.05}",60,0.951667,0.998333
8,"{'dense': 0.4, 'sparse': 0.15, 'bm25': 0.3, 'fts': 0.05, 'exact': 0.1}",60,0.951667,0.998333
9,"{'dense': 0.4, 'sparse': 0.2, 'bm25': 0.25, 'fts': 0.05, 'exact': 0.1}",60,0.951667,0.998333


Router TRAIN only: {'weights': {'dense': 0.4, 'sparse': 0.15, 'bm25': 0.35, 'fts': 0.05, 'exact': 0.05}, 'k': 60, 'section_accuracy_macro': 0.9566666666666667, 'section_recall@2_macro': 0.9983333333333334}


## 4. Chính sách pháp lý được giữ làm baseline
Cấu hình cũ được chọn trên 15 source-qrel development cases. Không chạy lại hoặc dùng bộ cũ làm điểm test hiện tại. Giữ trọng số cũ vì chưa có tập train pháp lý riêng; bộ 50 QA chỉ chạy sau khi khóa chính sách, không chọn trọng số trên QA50.

In [8]:
# Historical legal baseline remains separate from the newly trained QA policy.
import pandas as pd
from IPython.display import display

legal_policy = {**legal_policy, 'selection_status': 'LEGACY_DEVELOPMENT_POLICY_NOT_QA50_TUNED'}
display(pd.DataFrame([legal_policy]))
print('QA200 train selection and QA50 test are performed in the QA section after HS freeze.')

,weights,k,source_recall@5,mrr,selection_status
0,"{'dense': 0.05, 'sparse': 0.05, 'bm25': 0.15, 'fts': 0.05, 'exact': 0.7}",20,1.0,0.955556,LEGACY_DEVELOPMENT_POLICY_NOT_QA50_TUNED


QA200 train selection and QA50 test are performed in the QA section after HS freeze.


## Advanced retrieval and candidate-preserving pipeline — train only
Compare C11/C12 on several weight policies, then compare direct, hierarchy with descendant-supported parents, and dual paths.
Beam 5/10/20; branch pools 50/100; equal versus router-weighted Section fusion.
For candidate pools 20/30/50, compare cross-encoder and ColBERT, with reranker rank blending 0.5/0.75/1.0.
Choose per predicted Top-1 Section on train; no true Section is supplied to inference.
Notes/GIR are evidence, not an implemented legal exclusion reasoner.

Section columns in isolated train-ranking tables are not routing measurements; use the router table and test pipeline tables for Section accuracy.

In [9]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
reranker = CrossEncoder(cfg["reranker_model"], cfg["reranker_revision"], device=device)
print("Cross-encoder:", cfg["reranker_model"], cfg["reranker_revision"], device, flush=True)


def reranked(item, policy, kind, pool=30):
    rows = unique_codes(fuse({b: item["branches"][b] for b in policy["weights"]},
                             policy["k"], policy["weights"]), pool)
    return rerank_candidates(item["case"]["query"], rows,
        "cross_encoder" if kind == "C11" else "colbert", embedder, reranker, pool)[:10]

advanced_policies = {}
advanced_dev = []
for section in ("XIV", "XVI", "XVII"):
    subset = [x for x in dev_cache if x["case"]["section"] == section]
    for kind, base in [("C11", "C10"), ("C12", "C5")]:
        trials = []
        for policy in section_sweeps[section][base][:3]:
            with capture_output() as progress:
                predictions = [reranked(item, policy, kind) for item in subset]
            trial = {"section": section, "configuration": kind,
                "weights": policy["weights"], "k": policy["k"],
                **ranking_metrics([x["case"] for x in subset], predictions)}
            trials.append(trial)
            print("TRAIN advanced", section, kind, objective(trial), flush=True)
        advanced_policies[(section, kind)] = max(trials, key=objective)
        advanced_dev.extend(trials)
display(pd.DataFrame(advanced_dev))

router_train = {item["case"]["test_id"]: retriever.section_candidates(
    fuse(item["branches"], router_policy["k"], router_policy["weights"]))
    for item in dev_router_cache}
PIPELINE_GRID = []
for pool in (50, 100):
    for mixing in ("equal", "router"):
        PIPELINE_GRID.append({"strategy": "direct", "candidate_pool": pool,
            "section_mix": mixing, "beam_size": 10, "k": 20, "reranker": "none"})
        for beam in (5, 10, 20):
            PIPELINE_GRID.append({"strategy": "hierarchy", "candidate_pool": pool,
                "section_mix": mixing, "beam_size": beam, "k": 20, "reranker": "none"})
            for ratio in (0.25, 0.5, 0.75):
                PIPELINE_GRID.append({"strategy": "dual", "candidate_pool": pool,
                    "section_mix": mixing, "beam_size": beam, "direct_weight": ratio,
                    "k": 20, "reranker": "none"})

pipeline_train_trials = []
pipeline_policies = {}
for routed_section in ("XIV", "XVI", "XVII"):
    subset = [x for x in dev_cache if router_train[x["case"]["test_id"]]
              and router_train[x["case"]["test_id"]][0]["section"] == routed_section]
    candidates = []
    for params in PIPELINE_GRID:
        predictions = [candidate_paths(x["branches"], router_train[x["case"]["test_id"]],
                                        section_policies, params)["candidates"][:10] for x in subset]
        metric = ranking_metrics([x["case"] for x in subset], predictions)
        candidates.append({"routed_section": routed_section, "params": params, **metric})
    best_per_strategy = [max([r for r in candidates if r["params"]["strategy"] == strategy],
                             key=objective) for strategy in ("direct", "hierarchy", "dual")]
    for base in best_per_strategy:
        for kind in ("cross_encoder", "colbert"):
            # Pool 50 populates reusable per-pair scores once. Smaller pools then reuse them.
            for pool in (50, 30, 20):
                for blend in (1.0, 0.75, 0.5):
                    params = {**base["params"], "reranker": kind,
                              "rerank_pool": pool, "rank_weight": blend}
                    with capture_output() as progress:
                        predictions = [rerank_candidates(
                            x["case"]["query"], candidate_paths(x["branches"],
                            router_train[x["case"]["test_id"]], section_policies, params)["candidates"],
                            kind, embedder, reranker, pool, blend, params["k"])[:10] for x in subset]
                    trial = {"routed_section": routed_section, "params": params,
                             **ranking_metrics([x["case"] for x in subset], predictions)}
                    candidates.append(trial)
                print("TRAIN pipeline", routed_section, base["params"]["strategy"],
                      kind, pool, flush=True)
    chosen = max(candidates, key=objective)
    pipeline_policies[routed_section] = chosen["params"]
    pipeline_train_trials.extend(candidates)
    print("TRAIN chosen pipeline", routed_section, chosen, flush=True)
display(pd.DataFrame(pipeline_train_trials))

frozen = {"snapshot": retriever.snapshot, "profile": retriever.profile,
    "gold_hash": digest(gold), "retrieval": cfg, "section_policies": section_policies,
    "router": router_policy, "legal": legal_policy,
    "advanced": {s + ":" + k: p for (s, k), p in advanced_policies.items()},
    "pipeline_policies": pipeline_policies,
    "objective": "TRAIN Hit@1 -> Recall@5 -> MRR -> nDCG; pipeline selected by predicted Section",
    "selection_scope": "USER_SUPPLIED_TRAIN_ONLY; TEST_UNTOUCHED_DURING_SELECTION"}
write_json(ROOT / "config/selected_retrieval.json", frozen)
frozen_hash = digest(frozen)
print("Frozen before TEST:", frozen_hash)

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 3227.15it/s]

Cross-encoder: BAAI/bge-reranker-v2-m3 953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e mps


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


TRAIN advanced XIV C11 (0.435, 0.785, 0.5794424603174604, 0.6477011425303977)


...

TRAIN pipeline XVII dual colbert 30


TRAIN pipeline XVII dual colbert 20


TRAIN chosen pipeline XVII {'routed_section': 'XVII', 'params': {'strategy': 'direct', 'candidate_pool': 50, 'section_mix': 'router', 'beam_size': 10, 'k': 20, 'reranker': 'none'}, 'section_correct': 0.0, 'section_recall@2': 0.0, 'hit@1': 0.5, 'mrr': 0.5823668714797747, 'ndcg@10': 0.624570772133342, 'chapter_recall@5': 0.989247311827957, 'heading_recall@5': 0.9623655913978495, 'hs6_recall@5': 0.7634408602150538, 'ahtn8_recall@5': 0.6935483870967742, 'count': 186, 'hierarchy_metric_mode': 'ANCESTOR_PROJECTION_OF_LEAF_RANKING', 'exclusion_annotated_count': 0, 'exclusion_violation_rate': None}


,routed_section,params,section_correct,section_recall@2,hit@1,mrr,ndcg@10,chapter_recall@5,heading_recall@5,hs6_recall@5,ahtn8_recall@5,count,hierarchy_metric_mode,exclusion_annotated_count,exclusion_violation_rate
0,XIV,"{'strategy': 'direct', 'candidate_pool': 50, 'section_mix': 'equal', 'beam_size': 10, 'k': 20, 'reranker': 'none'}",0.0,0.0,0.505155,0.592743,0.659532,1.000000,0.912371,0.829897,0.783505,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
1,XIV,"{'strategy': 'hierarchy', 'candidate_pool': 50, 'section_mix': 'equal', 'beam_size': 5, 'k': 20, 'reranker': 'none'}",0.0,0.0,0.505155,0.592743,0.659532,1.000000,0.912371,0.829897,0.783505,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
2,XIV,"{'strategy': 'dual', 'candidate_pool': 50, 'section_mix': 'equal', 'beam_size': 5, 'direct_weight': 0.25, 'k': 20, 'reranker': 'none'}",0.0,0.0,0.505155,0.592743,0.659532,1.000000,0.912371,0.829897,0.783505,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
3,XIV,"{'strategy': 'dual', 'candidate_pool': 50, 'section_mix': 'equal', 'beam_size': 5, 'direct_weight': 0.5, 'k': 20, 'reranker': 'none'}",0.0,0.0,0.505155,0.592743,0.659532,1.000000,0.912371,0.829897,0.783505,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
4,XIV,"{'strategy': 'dual', 'candidate_pool': 50, 'section_mix': 'equal', 'beam_size': 5, 'direct_weight': 0.75, 'k': 20, 'reranker': 'none'}",0.0,0.0,0.505155,0.592743,0.659532,1.000000,0.912371,0.829897,0.783505,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
5,XIV,"{'strategy': 'hierarchy', 'candidate_pool': 50, 'section_mix': 'equal', 'beam_size': 10, 'k': 20, 'reranker': 'none'}",0.0,0.0,0.505155,0.592743,0.659532,1.000000,0.912371,0.829897,0.783505,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
6,XIV,"{'strategy': 'dual', 'candidate_pool': 50, 'section_mix': 'equal', 'beam_size': 10, 'direct_weight': 0.25, 'k': 20, 'reranker': 'none'}",0.0,0.0,0.505155,0.592743,0.659532,1.000000,0.912371,0.829897,0.783505,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
7,XIV,"{'strategy': 'dual', 'candidate_pool': 50, 'section_mix': 'equal', 'beam_size': 10, 'direct_weight': 0.5, 'k': 20, 'reranker': 'none'}",0.0,0.0,0.505155,0.592743,0.659532,1.000000,0.912371,0.829897,0.783505,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
8,XIV,"{'strategy': 'dual', 'candidate_pool': 50, 'section_mix': 'equal', 'beam_size': 10, 'direct_weight': 0.75, 'k': 20, 'reranker': 'none'}",0.0,0.0,0.505155,0.592743,0.659532,1.000000,0.912371,0.829897,0.783505,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
9,XIV,"{'strategy': 'hierarchy', 'candidate_pool': 50, 'section_mix': 'equal', 'beam_size': 20, 'k': 20, 'reranker': 'none'}",0.0,0.0,0.505155,0.592743,0.659532,1.000000,0.912371,0.829897,0.783505,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None


Frozen before TEST: 330edbf318bc4f65e270825f828d2e4bd05fee9104e63cfe46e12af07ec5531e


## Supplied test: 90 products / 180 query variants
All configurations and pipeline variants below are fixed on train. Test results do not change the selection.
Compare C1–C15 (excluding nonexistent family labels), and frozen pipeline ablations, per Section and query variant.
Upper-level metrics in this table are ancestor projections of leaf rankings; actual traversal is measured separately.

In [10]:
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
with capture_output() as test_progress:
    test_cache = branch_cache(retriever, test_cases, filters={"levels": [8]}, pool=100)
    test_router_cache = branch_cache(retriever, test_cases,
        filters={"sections": ["XIV", "XVI", "XVII"]}, pool=100)
router_sections = {item["case"]["test_id"]: retriever.section_candidates(
    fuse(item["branches"], router_policy["k"], router_policy["weights"])) for item in test_router_cache}
test_results = []
failures = []
for section in ("XIV", "XVI", "XVII"):
    for variant in ("name_only", "name_and_spec"):
        subset = [x for x in test_cache if x["case"]["section"] == section
                  and x["case"]["variant"] == variant]
        for family in [*CONFIGURATIONS, "C11", "C12"]:
            policy = section_sweeps[section][family][0] if family in CONFIGURATIONS else advanced_policies[(section, family)]
            with capture_output() as progress:
                predictions = [unique_codes(fuse({b: x["branches"][b] for b in policy["weights"]},
                    policy["k"], policy["weights"]), 10) if family in CONFIGURATIONS
                    else reranked(x, policy, family) for x in subset]
            metric = ranking_metrics([x["case"] for x in subset], predictions,
                [router_sections[x["case"]["test_id"]] for x in subset])
            test_results.append({"section": section, "variant": variant,
                "configuration": family, "weights": policy["weights"], "k": policy["k"], **metric})
            for item, ranked in zip(subset, predictions, strict=True):
                if item["case"]["expected_hs_8"] not in [r["code"] for r in ranked[:5]]:
                    failures.append({"section": section, "variant": variant, "configuration": family,
                        "id": item["case"]["test_id"], "expected": item["case"]["expected_hs_8"],
                        "returned": [r["code"] for r in ranked]})
print("C-family test evaluation complete; policies unchanged", flush=True)
display(pd.DataFrame(test_results))
display(pd.DataFrame(failures))

pipeline_test_results = []
pipeline_case_rows = []
for ablation in ("direct", "hierarchy", "dual", "selected"):
    predictions = []
    for item in test_cache:
        section_candidates = router_sections[item["case"]["test_id"]]
        params = dict(pipeline_policies[section_candidates[0]["section"]])
        if ablation != "selected":
            params.update(strategy=ablation, reranker="none")
        output = candidate_paths(item["branches"], section_candidates, section_policies, params)
        with capture_output() as progress:
            ranked = rerank_candidates(item["case"]["query"], output["candidates"],
                params.get("reranker", "none"), embedder, reranker,
                params.get("rerank_pool", 30), params.get("rank_weight", 1.0), params["k"])[:10]
        predictions.append(ranked)
        if ablation == "selected":
            expected = item["case"]["expected_hs_8"]
            codes = [r["code"] for r in ranked]
            pipeline_case_rows.append({"id": item["case"]["test_id"],
                "section": item["case"]["section"], "variant": item["case"]["variant"],
                "expected": expected, "rank": codes.index(expected) + 1 if expected in codes else None,
                "returned": codes, "policy_section": section_candidates[0]["section"],
                "trace": output["hierarchy"]})
    for section in ("XIV", "XVI", "XVII"):
        for variant in ("name_only", "name_and_spec"):
            indices = [i for i, x in enumerate(test_cache) if x["case"]["section"] == section
                       and x["case"]["variant"] == variant]
            metric = ranking_metrics([test_cache[i]["case"] for i in indices],
                [predictions[i] for i in indices],
                [router_sections[test_cache[i]["case"]["test_id"]] for i in indices])
            pipeline_test_results.append({"section": section, "variant": variant,
                                          "pipeline": ablation, **metric})
display(pd.DataFrame(pipeline_test_results))
display(pd.DataFrame(pipeline_case_rows))
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
print("No TEST-driven configuration edits. Exclusion violation rate: N/A without forbidden-code labels.")

C-family test evaluation complete; policies unchanged


,section,variant,configuration,weights,k,section_correct,section_recall@2,hit@1,mrr,ndcg@10,chapter_recall@5,heading_recall@5,hs6_recall@5,ahtn8_recall@5,count,hierarchy_metric_mode,exclusion_annotated_count,exclusion_violation_rate
0,XIV,name_only,C1,{'dense': 1.0},5,0.966667,1.0,0.100000,0.223624,0.326647,1.000000,0.766667,0.600000,0.466667,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
1,XIV,name_only,C2,{'sparse': 1.0},5,0.966667,1.0,0.066667,0.256296,0.354253,1.000000,0.700000,0.633333,0.533333,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
2,XIV,name_only,C3,{'bm25': 1.0},5,0.966667,1.0,0.066667,0.164074,0.228171,1.000000,0.466667,0.433333,0.333333,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
3,XIV,name_only,C4,{'fts': 1.0},5,0.966667,1.0,0.000000,0.035648,0.071684,0.366667,0.200000,0.033333,0.033333,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
4,XIV,name_only,C5,"{'dense': 0.4, 'sparse': 0.6}",10,0.966667,1.0,0.100000,0.268188,0.356370,1.000000,0.633333,0.633333,0.500000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
5,XIV,name_only,C6,"{'dense': 0.4104011295335827, 'bm25': 0.5895988704664173}",10,0.966667,1.0,0.066667,0.181151,0.249910,1.000000,0.533333,0.466667,0.366667,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
6,XIV,name_only,C7,"{'dense': 0.4659134648887171, 'sparse': 0.47803229031316746, 'fts': 0.056054244798115445}",10,0.966667,1.0,0.133333,0.278373,0.363213,1.000000,0.633333,0.600000,0.533333,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
7,XIV,name_only,C8,"{'dense': 0.3545648757459851, 'sparse': 0.5318745933303629, 'exact': 0.113560530923652}",60,0.966667,1.0,0.100000,0.289484,0.373613,1.000000,0.633333,0.633333,0.500000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
8,XIV,name_only,C9,"{'dense': 0.35, 'bm25': 0.35, 'exact': 0.3}",20,0.966667,1.0,0.066667,0.191706,0.272256,1.000000,0.533333,0.466667,0.400000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
9,XIV,name_only,C10,"{'dense': 0.2362063930546834, 'sparse': 0.04719840802210506, 'bm25': 0.538615980853112, 'fts': 0.16652548354897692, 'exact': 0.011453734521122722}",5,0.966667,1.0,0.066667,0.170093,0.240697,1.000000,0.500000,0.466667,0.366667,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None


,section,variant,configuration,id,expected,returned
0,XIV,name_only,C1,XIV_TE001:name_only,71131190,"[85393190, 71110010, 71159020, 85064000, 71070000, 71159010, 71131110, 71069100, 71141100, 71162000]"
1,XIV,name_only,C1,XIV_TE005:name_only,71131190,"[71171110, 71171190, 71110010, 71069200, 71129910, 71181010, 71069100, 71061000, 71142000, 71141100]"
2,XIV,name_only,C1,XIV_TE006:name_only,71131190,"[71131110, 71142000, 71141100, 71141900, 71110010, 71129910, 71131190, 71070000, 71171110, 71069100]"
3,XIV,name_only,C1,XIV_TE007:name_only,71131190,"[85064000, 71061000, 84789000, 71141100, 71110010, 71141900, 71142000, 71069200, 71159020, 71129910]"
4,XIV,name_only,C1,XIV_TE008:name_only,71131190,"[85064000, 71061000, 71141100, 71069200, 71141900, 71129910, 71151000, 71142000, 71110010, 71104900]"
5,XIV,name_only,C1,XIV_TE009:name_only,71131190,"[85064000, 71159020, 71159010, 71142000, 71070000, 71141100, 71131110, 71110010, 71141900, 71069200]"
6,XIV,name_only,C1,XIV_TE010:name_only,71131190,"[71179094, 85064000, 71039190, 71129910, 71061000, 71141100, 71131190, 71039110, 71142000, 71151000]"
7,XIV,name_only,C1,XIV_TE011:name_only,71131990,"[71159020, 71131190, 71131110, 71159010, 71179094, 71162000, 71131990, 71131910, 71141900, 71171920]"
8,XIV,name_only,C1,XIV_TE012:name_only,71131990,"[71159020, 71159010, 71131110, 71131190, 71179094, 71162000, 71081300, 71131910, 71131990, 71132090]"
9,XIV,name_only,C1,XIV_TE013:name_only,71131990,"[71159020, 71159010, 71131190, 71131110, 71132090, 71179094, 71131990, 71171920, 71131910, 71179020]"


,section,variant,pipeline,section_correct,section_recall@2,hit@1,mrr,ndcg@10,chapter_recall@5,heading_recall@5,hs6_recall@5,ahtn8_recall@5,count,hierarchy_metric_mode,exclusion_annotated_count,exclusion_violation_rate
0,XIV,name_only,direct,0.966667,1.0,0.066667,0.180873,0.242469,1.0,0.533333,0.466667,0.400000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
1,XIV,name_and_spec,direct,0.933333,1.0,0.033333,0.202540,0.275644,1.0,0.500000,0.466667,0.433333,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
2,XVI,name_only,direct,1.000000,1.0,0.900000,0.925595,0.942658,1.0,0.966667,0.966667,0.933333,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
3,XVI,name_and_spec,direct,1.000000,1.0,0.966667,0.977778,0.983333,1.0,1.000000,1.000000,1.000000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
4,XVII,name_only,direct,0.966667,1.0,0.300000,0.456243,0.538904,1.0,0.933333,0.800000,0.700000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
5,XVII,name_and_spec,direct,0.933333,1.0,0.300000,0.468968,0.578719,1.0,0.966667,0.800000,0.700000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
6,XIV,name_only,hierarchy,0.966667,1.0,0.066667,0.180873,0.242469,1.0,0.533333,0.466667,0.400000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
7,XIV,name_and_spec,hierarchy,0.933333,1.0,0.033333,0.202540,0.275644,1.0,0.500000,0.466667,0.433333,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
8,XVI,name_only,hierarchy,1.000000,1.0,0.900000,0.925595,0.942658,1.0,0.966667,0.966667,0.933333,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
9,XVI,name_and_spec,hierarchy,1.000000,1.0,0.966667,0.977778,0.983333,1.0,1.000000,1.000000,1.000000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None


,id,section,variant,expected,rank,returned,policy_section,trace
0,XIV_TE001:name_only,XIV,name_only,71131190,NaN,"[71162000, 85064000, 71141100, 85393190, 71159020, 84283390, 71161000, 84284000, 71110010, 84283310]",XIV,"[{'section': 'XIV', 'stages': {'chapter': ['71'], 'heading': ['7116', '7114', '7115', '7111', '7106', '7113', '7107', '7118', '7112', '7117'], 'hs6': ['711620', '711411', '711590', '711610', '711100', '711420', '711419', '710691', '711311', '710700'], 'ahtn8': ['71162000', '71141100', '71159020', '71161000', '71110010', '71142000', '71141900', '71159010', '71069100', '71131110', '71070000', '71131190', '71110090']}, 'parent_scoring': 'DESCENDANT_RANK_SUPPORT; not synthetic legal text'}, {'section': 'XVI', 'stages': {'chapter': ['85', '84'], 'heading': ['8506', '8539', '8428', '8518', '8409', '8419', '8456', '8519'], 'hs6': ['850640', '853931', '842833', '842840', '842870', '842839', '851830', '842890', '842831', '842832'], 'ahtn8': ['85064000', '85393190', '84283390', '84284000', '84283310', '84287000', '84283990', '85183051', '84289090', '84283100', '84283290', '85183010', '84283910', '85183020', '85183040', '84283210', '85183059', '84289020', '85183090', '84289030', '85393110', '85393120']}, 'parent_scoring': 'DESCENDANT_RANK_SUPPORT; not synthetic legal text'}]"
1,XIV_TE001:name_and_spec,XIV,name_and_spec,71131190,10.0,"[71162000, 71141100, 85064000, 71159020, 84283390, 71159010, 85393200, 71141900, 84284000, 71131190]",XIV,"[{'section': 'XIV', 'stages': {'chapter': ['71'], 'heading': ['7116', '7114', '7115', '7113', '7118', '7111', '7107', '7106', '7108', '7112'], 'hs6': ['711620', '711411', '711590', '711419', '711311', '711610', '711890', '711100', '711420', '711810'], 'ahtn8': ['71162000', '71141100', '71159020', '71159010', '71141900', '71131190', '71161000', '71131110', '71189020', '71110010', '71142000', '71181010', '71110090', '71189010']}, 'parent_scoring': 'DESCENDANT_RANK_SUPPORT; not synthetic legal text'}, {'section': 'XVI', 'stages': {'chapter': ['85', '84'], 'heading': ['8506', '8428', '8539', '8518', '8482', '8409', '8484', '8519', '8546'], 'hs6': ['850640', '842833', '853932', '842840', '842870', '842839', '851830', '842890', '853931', '842831'], 'ahtn8': ['85064000', '84283390', '85393200', '84284000', '84283310', '84287000', '84283990', '85183051', '84289090', '85393190', '84283100', '85183010', '84283910', '85183020', '85183040', '85183059', '85183090', '84289020', '84289030', '85393110', '85393120']}, 'parent_scoring': 'DESCENDANT_RANK_SUPPORT; not synthetic legal text'}]"
2,XIV_TE002:name_only,XIV,name_only,71131190,4.0,"[85064000, 71141100, 71159010, 71131190, 71110010, 71142000, 71069100, 71141900, 71061000, 71159020]",XVI,"[{'section': 'XVI', 'stages': {'chapter': ['85', '84'], 'heading': ['8506', '8544', '8456', '8539', '8515', '8480', '8484', '8547', '8409', '8514'], 'hs6': ['850640', '854411', '845640', '853941', '853921', '853990', '854449', '853910', '853949', '853951'], 'ahtn8': ['85064000', '85441190', '85441140', '85441130', '84564010', '85394100', '85392130', '85399020', '85444933', '85441120', '85444932', '85444922', '85391010', '85444942', '85394900', '85444924', '85399010', '85444913', '85395100', '85444929', '85444949', '85444919', '85392190', '85444939', '85399030', '84564090', '85392140', '85391090', '85392120', '84564020']}, 'parent_scoring': 'DESCENDANT_RANK_SUPPORT; not synthetic legal text'}, {'section': 'XIV', 'stages': {'chapter': ['71'], 'heading': ['7114', '7115', '7111', '7113', '7106', '7118', '7107', '7116', '7109', '7117'], 'hs6': ['711411', '711420', '711419', '711590', '711100', '711311', '710691', '710610', '711890', '710692'], 'ahtn8': ['71141100', '71142000', '71141900', '71159020', '71159010', '71110010', '71131110', '71069100', '71131190', '71061000', '71189020', '71069200', '71110090', '71189010']}, 'parent_scoring': 'DESCENDANT_RANK_SUPPORT; not synthetic legal text'}]"
3,XIV_TE002:name_and_spec,XIV,name_and_spec,71131190,NaN,"[71141100, 71142000,

No TEST-driven configuration edits. Exclusion violation rate: N/A without forbidden-code labels.


## Strict legacy traversal versus selected adaptive pipeline
Run the original top-down algorithm on the supplied test with the newly train-selected base weights.
Store resumable cache only; detailed per-case failures and traces are rendered here.
Diagnose the earliest lost ancestor and distinguish pruning from final ranking losses.
Four curated cross-boundary cases are a separate regression check, not the gold exclusion metric.
**Lưu ý:** các output của các cell 18–23 là mốc lịch sử; cell 25–33 đã được chạy lại trên snapshot active, còn benchmark QA mới nằm ở phần QA của notebook 01.


In [11]:
import gzip
import json

MODE = {"C1": "dense", "C2": "sparse", "C3": "bm25", "C4": "fts",
    "C5": "dense_sparse", "C6": "dense_bm25", "C7": "hybrid_sparse",
    "C8": "dense_sparse_exact", "C9": "dense_bm25_exact", "C10": "full_hybrid",
    "C13": "full_hybrid", "C14": "full_hybrid", "C15": "full_hybrid"}
runtime_policies = {s: {**p, "mode": MODE[p["configuration"]]} for s, p in section_policies.items()}
traversal_rows = []
strict_failures = []
strict_cache = ROOT / "artifacts/strict_traversal_cache"
strict_cache.mkdir(exist_ok=True)
for case in test_cases:
    target = strict_cache / (digest({"snapshot": retriever.snapshot, "query": case["query"],
        "weights": section_policies, "router": router_policy, "version": "strict-v3"}) + ".json.gz")
    if target.exists():
        with gzip.open(target, "rt") as stream:
            result = json.load(stream)
    else:
        with capture_output() as progress:
            result = retriever.hierarchy_search(case["query"], weights=router_policy["weights"],
                section_policies=runtime_policies, rrf_k=router_policy["k"])
        with gzip.open(target, "wt") as stream:
            json.dump(result, stream, ensure_ascii=False, default=str)
    expected = case["expected_hs_8"]
    row = {"section": case["section"], "variant": case["variant"], "id": case["test_id"],
        "section_accuracy": bool(result["section_candidates"] and result["section_candidates"][0]["section"] == case["section"]),
        "section_recall@2": any(x["section"] == case["section"] for x in result["section_candidates"]),
        "hit@1": bool(result["candidates"] and result["candidates"][0]["code"] == expected)}
    lost = []
    for name, length in [("chapter", 2), ("heading", 4), ("hs6", 6), ("ahtn8", 8)]:
        hit = expected in [r["code"] for r in result["candidates"][:5]] if name == "ahtn8" else any(
            expected[:length] in stage["stages"][name][:5] for stage in result["hierarchy"])
        row[name + "_recall@5"] = hit
        if not hit:
            lost.append(name)
    traversal_rows.append(row)
    if lost:
        strict_failures.append({"id": case["test_id"], "query": case["query"], "expected": expected,
            "first_loss": lost[0], "trace": result["hierarchy"],
            "returned": [r["code"] for r in result["candidates"]]})
print("Strict test traversal complete", flush=True)
display(pd.DataFrame(traversal_rows).groupby(["section", "variant"]).mean(numeric_only=True))
display(pd.DataFrame(strict_failures))

# Reload the frozen runtime; this exercises the same candidate_paths and rerank helper used above.
runtime = Retriever(conn, embedder)
runtime._reranker = reranker
for section in ("XIV", "XVI", "XVII"):
    case = next(c for c in test_cases if c["section"] == section and c["variant"] == "name_and_spec")
    with capture_output() as progress:
        actual = runtime.routed_search(case["query"])
    expected = next(r for r in pipeline_case_rows if r["id"] == case["test_id"])
    assert [r["code"] for r in actual["candidates"]] == expected["returned"]
    print("Runtime/benchmark parity:", case["test_id"], actual["pipeline_policy"],
          actual["validation_status"])

# Curated cross-boundary qrels, grounded in the local Section XVII Note 2.
exclusion_cases = [
    {
        "query": "Máy phát điện xoay chiều dùng với động cơ đốt trong của ô tô",
        "expected_hs6": "851150",
        "forbidden_prefix": "87",
    },
    {
        "query": "Bugi đánh lửa dùng cho động cơ đốt trong của xe ô tô",
        "expected_hs6": "851110",
        "forbidden_prefix": "87",
    },
    {
        "query": "Thiết bị điện chiếu sáng và tín hiệu đèn pha dùng cho ô tô",
        "expected_hs6": "851220",
        "forbidden_prefix": "87",
    },
    {
        "query": "Cần gạt nước kính chắn gió vận hành bằng điện dùng cho ô tô",
        "expected_hs6": "851240",
        "forbidden_prefix": "87",
    },
]
exclusion_rows = []
for case in exclusion_cases:
    anchor = next(r for r in bundle["records"] if r["id"] == "tt31:SXVII_NOTE_2")
    assert "Máy điện hoặc thiết bị điện (Chương 85)" in anchor["source_text"]
    assert any(
        r.get("code", "").startswith(case["expected_hs6"])
        for r in bundle["records"]
        if r.get("code")
    )
    output = retriever.search(
        case["query"],
        mode="full_hybrid",
        levels=[8],
        top_k=5,
        expand=False,
        weights=router_policy["weights"],
    )
    codes = [r["code"] for r in output["candidates"]]
    exclusion_rows.append(
        {
            "query": case["query"],
            "expected_hs6": case["expected_hs6"],
            "returned": codes,
            "hs6_recall@5": any(c.startswith(case["expected_hs6"]) for c in codes),
            "exclusion_violation": any(c.startswith(case["forbidden_prefix"]) for c in codes),
            "source_note": anchor["id"],
        }
    )
display(pd.DataFrame(exclusion_rows))
print(
    "Curated 4-case retrieval Exclusion Violation Rate:",
    np.mean([r["exclusion_violation"] for r in exclusion_rows]),
)
print(
    "This is a source-grounded retrieval conflict suite, not the 390-product gold E.V.R. or final classification accuracy."
)

Strict test traversal complete


section_accuracy  section_recall@2     hit@1  \
section variant                                                       
XIV     name_and_spec          0.666667          0.966667  0.133333   
        name_only              0.733333          1.000000  0.133333   
XVI     name_and_spec          1.000000          1.000000  0.933333   
        name_only              1.000000          1.000000  0.800000   
XVII    name_and_spec          0.966667          1.000000  0.066667   
        name_only              0.966667          1.000000  0.133333   

                       chapter_recall@5  heading_recall@5  hs6_recall@5  \
section variant                                                           
XIV     name_and_spec          0.966667          0.366667      0.333333   
        name_only              1.000000          0.400000      0.333333   
XVI     name_and_spec          1.000000          0.933333      0.933333   
        name_only              1.000000          0.866667      0.866667   
XVII    name_and_spec          1.000000          0.800000      0.766667   
        name_only              0.966667          0.566667      0.566667   

                       ahtn8_recall@5  
section variant                        
XIV     name_and_spec        0.300000  
        name_only            0.300000  
XVI     name_and_spec        0.933333  
        name_only            0.866667  
XVII    name_and_spec        0.733333  
        name_only            0.500000

,id,query,expected,first_loss,trace,returned
0,XIV_TE001:name_only,Hoa tai bằng bạc,71131190,heading,"[{'section': 'XIV', 'stages': {'chapter': ['71'], 'heading': ['7106', '7114', '7111', '7116', '7117'], 'hs6': ['711411', '710692', '710691', '711420', '711100'], 'hs6_implicit_prefixes': [], 'ahtn8': ['71141100', '71110010', '71142000', '71069200', '71069100', '71110090']}}, {'section': 'XVI', 'stages': {'chapter': ['85', '84'], 'heading': ['8518', '8456', '8423', '8539', '8546'], 'hs6': ['851830', '853931', '853990', '853932', '851890'], 'hs6_implicit_prefixes': ['853931', '853932'], 'ahtn8': ['85183010', '85183020', '85183051', '85183040', '85189020', '85183059', '85189040', '85183090', '85189030', '85393190']}}]","[71141100, 85183010, 71110010, 85183020, 71142000, 85183051, 71069200, 85183040, 71069100, 85189020]"
1,XIV_TE001:name_and_spec,Hoa tai bằng bạc. Hoa tai bằng bạc,71131190,heading,"[{'section': 'XIV', 'stages': {'chapter': ['71'], 'heading': ['7106', '7114', '7111', '7116', '7108'], 'hs6': ['711411', '710692', '710691', '711420', '711419'], 'hs6_implicit_prefixes': [], 'ahtn8': ['71141100', '71141900', '71142000', '71069200', '71069100']}}, {'section': 'XVI', 'stages': {'chapter': ['85', '84'], 'heading': ['8518', '8456', '8526', '8546', '8423'], 'hs6': ['851830', '854610', '851829', '851840', '851890'], 'hs6_implicit_prefixes': ['854610'], 'ahtn8': ['85183010', '85183020', '85183051', '85183040', '85461000', '85183059', '85183090', '85184030', '85189020', '85189040']}}]","[71141100, 85183010, 71141900, 85183020, 71142000, 85183051, 71069200, 85183040, 71069100, 85461000]"
2,XIV_TE002:name_only,Dây đồng hồ bằng bạc,71131190,heading,"[{'section': 'XVI', 'stages': {'chapter': ['84', '85'], 'heading': ['8544', '8539', '8451', '8515', '8547'], 'hs6': ['854411', '854790', '853941', '854449', '853990'], 'hs6_implicit_prefixes': ['853941', '854449'], 'ahtn8': ['85441130', '85441140', '85441190', '85441120', '85399010', '85394100', '85479010', '85444932', '85444922', '85444933']}}, {'section': 'XIV', 'stages': {'chapter': ['71'], 'heading': ['7106', '7114', '7111', '7117', '7115'], 'hs6': ['711411', '711100', '710692', '710691', '711590'], 'hs6_implicit_prefixes': [], 'ahtn8': ['71141100', '71110010', '71159020', '71159010', '71069100', '71110090', '71069200', '71159090']}}]","[71141100, 85441130, 71110010, 85441140, 71159020, 85441190, 71159010, 85441120, 71069100, 85399010]"
3,XIV_TE002:name_and_spec,Dây đồng hồ bằng bạc. Dây đồng hồ bằng bạc,71131190,heading,"[{'section': 'XVI', 'stages': {'chapter': ['85', '84'], 'heading': ['8544', '8539', '8451', '8515', '8484'], 'hs6': ['854411', '848490', '848410', '853941', '853939'], 'hs6_implicit_prefixes': ['848490', '848410', '853941'], 'ahtn8': ['85441130', '85441140', '85441190', '85394100', '85441120', '84841000', '85393990', '85393910', '85393940', '84849000']}}, {'section': 'XIV', 'stages': {'chapter': ['71'], 'heading': ['7106', '7114', '7111', '7118', '7117'], 'hs6': ['711411', '710692', '710691', '711890', '711420'], 'hs6_implicit_prefixes': [], 'ahtn8': ['71141100', '71189020', '71069200', '71069100', '71142000', '71189090', '71189010']}}]","[71141100, 85441130, 71189020, 85441140, 71069200, 85441190, 71069100, 85394100, 71142000, 85441120]"
4,XIV_TE004:name_only,Mặt dây chuyền bằng bạc,71131190,heading,"[{'section': 'XVI', 'stages': {'chapter': ['85', '84'], 'heading': ['8462', '8539', '8547', '8544', '8447'], 'hs6': ['846232', '846239', '853931', '846233', '853990'], 'hs6_implicit_prefixes': ['846239', '853931', '846233'], 'ahtn8': ['84623210', '84623290', '85393190', '85399030', '85399010', '84623900', '85393120', '84623300', '85399020', '85393110']}}, {'section': 'XIV', 'stages': {'chapter': ['71'], 'heading': ['7106', '7114', '7111', '7117', '7115'], 'hs6': ['711411', '710691', '710692', '711590', '711100'], 'hs6_implicit_prefixes': [], 'ahtn8': ['71141100', '71110010', '71159020', '71159010', '71069200', '71069100', '71110090', '71159090']}}]","[71141100, 84623

Runtime/benchmark parity: XIV_TE001:name_and_spec {'strategy': 'direct', 'candidate_pool': 50, 'section_mix': 'router', 'beam_size': 10, 'k': 20, 'reranker': 'none'} REQUIRES_GIR_EXCLUSION_REASONING


Runtime/benchmark parity: XVI_TE001:name_and_spec {'strategy': 'direct', 'candidate_pool': 100, 'section_mix': 'router', 'beam_size': 10, 'k': 20, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.75} REQUIRES_GIR_EXCLUSION_REASONING


Runtime/benchmark parity: XVII_TE001:name_and_spec {'strategy': 'direct', 'candidate_pool': 100, 'section_mix': 'router', 'beam_size': 10, 'k': 20, 'reranker': 'none'} REQUIRES_GIR_EXCLUSION_REASONING


,query,expected_hs6,returned,hs6_recall@5,exclusion_violation,source_note
0,Máy phát điện xoay chiều dùng với động cơ đốt trong của ô tô,851150,"[85115031, 85115032, 85115033, 85115099, 85115091]",True,False,tt31:SXVII_NOTE_2
1,Bugi đánh lửa dùng cho động cơ đốt trong của xe ô tô,851110,"[85111020, 85111010, 85111090, 85112010, 85113030]",True,False,tt31:SXVII_NOTE_2
2,Thiết bị điện chiếu sáng và tín hiệu đèn pha dùng cho ô tô,851220,"[85121000, 85122091, 85122020, 85122099, 85123099]",True,False,tt31:SXVII_NOTE_2
3,Cần gạt nước kính chắn gió vận hành bằng điện dùng cho ô tô,851240,"[85124000, 85122091, 85123010, 85121000, 85123099]",True,False,tt31:SXVII_NOTE_2


Curated 4-case retrieval Exclusion Violation Rate: 0.0
This is a source-grounded retrieval conflict suite, not the 390-product gold E.V.R. or final classification accuracy.


## Local uncached query latency
Report C1–C15 and selected adaptive pipeline. 3 samples per Section are a local check, not a production SLO.
Passage vectors and corpus indexes are prebuilt; bypass query/model score caches during latency measurement.

Single-family timings use branch pool50; the selected pipeline uses its trained pool50/100. Timings exclude Notes/GIR/graph expansion and model cold start.

In [12]:
latency_rows = []
for section in ("XIV", "XVI", "XVII"):
    subset = [x for x in test_cache if x["case"]["section"] == section and x["case"]["variant"] == "name_and_spec"][:3]
    for family in [*CONFIGURATIONS, "C11", "C12", "selected_pipeline"]:
        for item in subset:
            query = item["case"]["query"]
            start = time.perf_counter()
            with capture_output() as progress:
                if family == "selected_pipeline":
                    # Fresh branch/query inference, then the trained pipeline with uncached reranking.
                    fresh = branch_cache(retriever, [item["case"]], filters={"levels": [8]}, pool=100, use_cache=False)[0]
                    router_fresh = branch_cache(retriever, [item["case"]],
                        filters={"sections": ["XIV", "XVI", "XVII"]}, pool=100, use_cache=False)[0]
                    sections = retriever.section_candidates(fuse(router_fresh["branches"], router_policy["k"], router_policy["weights"]))
                    params = pipeline_policies[sections[0]["section"]]
                    rows = candidate_paths(fresh["branches"], sections, section_policies, params)["candidates"]
                    rerank_candidates(query, rows, params["reranker"], embedder, reranker,
                        params.get("rerank_pool", 30), params.get("rank_weight", 1), params["k"], use_cache=False)
                else:
                    policy = section_sweeps[section][family][0] if family in CONFIGURATIONS else advanced_policies[(section, family)]
                    mode = MODE[family] if family in CONFIGURATIONS else "full_hybrid" if family == "C11" else "dense_sparse"
                    output = retriever.search(query, mode=mode, levels=[8], top_k=30,
                        expand=False, weights=policy["weights"], rrf_k=policy["k"], use_cache=False)
                    if family in ("C11", "C12"):
                        rerank_candidates(query, output["candidates"], "cross_encoder" if family == "C11" else "colbert",
                            embedder, reranker, use_cache=False)
            latency_rows.append({"section": section, "configuration": family,
                "milliseconds": (time.perf_counter() - start) * 1000})
display(pd.DataFrame(latency_rows).groupby(["section", "configuration"]).agg(
    samples=("milliseconds", "count"), p50_ms=("milliseconds", "median"),
    p95_ms=("milliseconds", lambda s: s.quantile(0.95))))
print("Fresh query inference / database branches / candidate paths / rerank; excludes Notes/GIR/graph expansion and model cold start.")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


samples       p50_ms       p95_ms
section configuration                                       
XIV     C1                       3   118.790292   179.487604
        C10                      3   139.695000   144.392962
        C11                      3  1558.777917  3625.878679
        C12                      3   236.598792   266.359992
        C13                      3   135.587459   145.608883
        C14                      3   136.931708   140.667795
        C15                      3   132.294417   141.489642
        C2                       3    98.150041   104.264267
        C3                       3     8.679666     9.104842
        C4                       3    16.192917    23.171180
        C5                       3   117.777208   123.600509
        C6                       3   117.447291   120.093704
        C7                       3   131.922542   134.720341
        C8                       3   115.635666   122.512717
        C9                       3   116.422958   120.238245
        selected_pipeline        3   659.837917  1565.656005
XVI     C1                       3   103.953334   121.554747
        C10                      3   147.445209   181.136858
        C11                      3  1189.978667  1380.895629
        C12                      3   201.980750   247.749312
        C13                      3   137.383583   181.258621
        C14                      3   136.930500   179.490600
        C15                      3   135.983083   183.562334
        C2                       3    87.804583   110.529808
        C3                       3    11.961458    14.362096
        C4                       3    27.126583    31.536396
        C5                       3   104.467416   128.976517
        C6                       3   102.364708   129.869946
        C7                       3   124.255750   162.787112
        C8                       3   105.959875   129.313449
        C9                       3   100.091250   128.855887
        selected_pipeline        3  1173.032917  1483.811317
XVII    C1                       3   152.023750   152.213799
        C10                      3   232.046125   389.237501
        C11                      3  1393.432375  1450.620887
        C12                      3   306.646750   311.067513
        C13                      3   224.443542   389.367529
        C14                      3   234.728666   390.417866
        C15                      3   231.106792   388.569554
        C2                       3   143.365750   145.549749
        C3                       3    16.582292    17.955391
        C4                       3    34.231459    53.833797
        C5                       3   156.751959   161.164546
        C6                       3   167.512792   168.182430
        C7                       3   197.648083   198.887721
        C8                       3   172.456791   325.890630
        C9                       3   170.228750   321.490175
        selected_pipeline        3   742.911083  1116.776670

Fresh query inference / database branches / candidate paths / rerank; excludes Notes/GIR/graph expansion and model cold start.


In [5]:
import subprocess

env = {**os.environ, "RAG_RUN_DB_TESTS": "1", "TOKENIZERS_PARALLELISM": "false"}
for command in [
    [str(ROOT / ".venv/bin/ruff"), "check", "rag", "tests", "notebooks"],
    [str(ROOT / ".venv/bin/mypy"), "rag"],
    [str(ROOT / ".venv/bin/bandit"), "-r", "rag", "-ll", "-q"],
    [str(ROOT / ".venv/bin/python"), "-m", "pytest", "tests", "-q"],
]:
    result = subprocess.run(command, cwd=ROOT, env=env, capture_output=True, text=True, check=False)
    print(" ".join(command[1:]), "=>", result.returncode, result.stdout or result.stderr or "PASS")
    assert result.returncode == 0
assert not conn.closed


check rag tests notebooks => 0 All checks passed!



rag => 0 Success: no issues found in 6 source files



-r rag -ll -q => 0 PASS


-m pytest tests -q => 0 ......................................                                   [100%]
38 passed in 20.73s



## Interpretation
Use test tables to describe generalization, not to retune. Hit@1 is first-candidate retrieval accuracy, not a legal classification conclusion.
Keep supplied train/test split (8 overlapping labels), cleaned queries, pinned model revisions and library implementations.
Runtime uses trained section/router/pipeline policies. Notes/GIR evidence retrieval does not execute exclusions or replace a Critic/Human-in-the-Loop.
QA50 replaces the old 15-case retrieval test and is evaluated after the final policy freeze.

In [6]:
from io import StringIO

import nbformat
import pandas as pd

# Summarize the actual executed tables; numeric outputs remain in their original cells.
completed = nbformat.read(ROOT / "notebooks/02_test_retrieval.ipynb", as_version=4)
def saved_tables(cell):
    return [pd.read_html(StringIO(o.data["text/html"]))[0]
            for o in completed.cells[cell].outputs
            if o.output_type in ("display_data", "execute_result") and "text/html" in o.data]

pipeline = next(t for t in saved_tables(16) if "pipeline" in t)
selected = pipeline[pipeline.pipeline == "selected"].copy()
legacy = saved_tables(18)[0]
legacy.columns = [c[1] if c[0].startswith("Unnamed") else c[0] for c in legacy.columns]
summary = selected[["section", "variant", "hit@1", "ahtn8_recall@5", "section_recall@2"]].merge(
    legacy[["section", "variant", "hit@1", "ahtn8_recall@5"]],
    on=["section", "variant"], suffixes=("_selected", "_strict"))
display(summary)
print("Policies were frozen on 300 TRAIN products; these metrics describe 90 TEST products.")
print("XIV remains weak. TEST labels were not used to change weights or rerank policies.")
print("QA50 is a new mixed-source, multi-level held-out benchmark; evaluated after the final freeze below.")


,section,variant,hit@1_selected,ahtn8_recall@5_selected,section_recall@2,hit@1_strict,ahtn8_recall@5_strict
0,XIV,name_only,0.233333,0.600000,1.0,0.133333,0.300000
1,XIV,name_and_spec,0.166667,0.566667,1.0,0.133333,0.300000
2,XVI,name_only,0.900000,1.000000,1.0,0.800000,0.866667
3,XVI,name_and_spec,1.000000,1.000000,1.0,0.933333,0.933333
4,XVII,name_only,0.466667,1.000000,1.0,0.133333,0.500000
5,XVII,name_and_spec,0.466667,0.933333,1.0,0.066667,0.733333


Policies were frozen on 300 TRAIN products; these metrics describe 90 TEST products.
XIV remains weak. TEST labels were not used to change weights or rerank policies.
QA50 is a new mixed-source, multi-level held-out benchmark; evaluated after the final freeze below.


## Definition-aware retrieval improvement — train selection, then diagnostic test
Earlier outputs remain as the baseline. TEST was inspected during diagnosis, so new TEST scores are **diagnostic regression results**, not a new untouched holdout.
Search views follow positive definition edges and CONTAINS descendants. Reference placeholders resolve only to exact source-parent excerpts. Exclusions never generate positive candidates. Original chunks and BGE vectors are unchanged; no gold is ingested.
Tune only on TRAIN by predicted Section, then freeze before diagnostic TEST. Evidence-aware reranking is compared separately.

In [4]:
from rag.retrieval import definition_fusion

baseline_selection = read(ROOT / "config/selected_retrieval.json")
section_policies = baseline_selection["section_policies"]
router_policy = baseline_selection["router"]
baseline_pipeline = baseline_selection["pipeline_policies"]
assert not conn.closed
views = retriever.definition_views()
print("Graph-derived leaf views:", len(views))
display(pd.DataFrame([{"section": v["record"]["section"],
    "heading": v["record"]["heading"], "candidate": v["record"]["code"],
    "note_ids": [x["id"] for x in v["citations"]], "paths": v["paths"]}
    for v in views.values()]))
with capture_output() as progress:
    definition_train_cache = branch_cache(retriever, dev_cases, filters={"levels": [8]}, pool=100)
    definition_router_train_cache = branch_cache(retriever, dev_cases,
        filters={"sections": ["XIV", "XVI", "XVII"]}, pool=100)
train_sections = {x["case"]["test_id"]: retriever.section_candidates(
    fuse(x["branches"], router_policy["k"], router_policy["weights"]))
    for x in definition_router_train_cache}
train_definitions = {x["case"]["test_id"]: retriever.definition_candidates(
    x["case"]["query"], 100, [s["section"] for s in train_sections[x["case"]["test_id"]]])
    for x in definition_train_cache}
train_direct = {x["case"]["test_id"]: candidate_paths(x["branches"],
    train_sections[x["case"]["test_id"]], section_policies,
    baseline_pipeline[train_sections[x["case"]["test_id"]][0]["section"]])["candidates"]
    for x in definition_train_cache}

def definition_objective(row):
    return row["hit@1"], row["ahtn8_recall@5"], row["mrr"], row["ndcg@10"]

# Selection uses TRAIN only.
definition_trials = []
definition_shortlist = {}
for section in ("XIV", "XVI", "XVII"):
    subset = [x for x in definition_train_cache
        if train_sections[x["case"]["test_id"]][0]["section"] == section]
    trials = []
    for weight in np.linspace(0, 1, 21):
        for k in (5, 10, 20, 40, 60, 100):
            rankings = [definition_fusion(train_direct[x["case"]["test_id"]],
                train_definitions[x["case"]["test_id"]], float(weight), k)[:10] for x in subset]
            metric = ranking_metrics([x["case"] for x in subset], rankings)
            trials.append({"routed_section": section, "definition_weight": float(weight),
                "k": k, "reranker": "none", **metric})
    trials.sort(key=definition_objective, reverse=True)
    definition_shortlist[section] = trials[:3]
    definition_trials.extend(trials)
    print("TRAIN best definition branch", section, trials[0], flush=True)
display(pd.DataFrame(definition_trials).sort_values(
    ["routed_section", "hit@1", "ahtn8_recall@5", "mrr"], ascending=[True, False, False, False]))


Graph-derived leaf views: 49


,section,heading,candidate,note_ids,paths
0,XIV,7113,71132090,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71132090'}]"
1,XIV,7113,71132010,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71132010'}]"
2,XIV,7113,71131990,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131990'}]"
3,XIV,7113,71131910,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131910'}]"
4,XIV,7113,71131190,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]"
5,XIV,7113,71131110,[tt31:XIV_CH71_NOTE_09],"[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131110'}]"
6,XIV,7114,71142000,[tt31:XIV_CH71_NOTE_10],"[{'note_id': 'tt31:XIV_CH71_NOTE_10', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7114', 'candidate_id': 'tt31:HS8:71142000'}]"
7,XIV,7114,71141900,[tt31:XIV_CH71_NOTE_10],"[{'note_id': 'tt31:XIV_CH71_NOTE_10', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7114', 'candidate_id': 'tt31:HS8:71141900'}]"
8,XIV,7114,71141100,[tt31:XIV_CH71_NOTE_10],"[{'note_id': 'tt31:XIV_CH71_NOTE_10', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7114', 'candidate_id': 'tt31:HS8:71141100'}]"
9,XIV,7117,71179099,"[tt31:XIV_CH71_NOTE_11, tt31:XIV_CH71_NOTE_09]","[{'note_id': 'tt31:XIV_CH71_NOTE_11', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7117', 'candidate_id': 'tt31:HS8:71179099'}]"


TRAIN best definition branch XIV {'routed_section': 'XIV', 'definition_weight': 0.1, 'k': 5, 'reranker': 'none', 'section_correct': 0.0, 'section_recall@2': 0.0, 'hit@1': 0.5051546391752577, 'mrr': 0.6482715594828997, 'ndcg@10': 0.7175563413046115, 'chapter_recall@5': 1.0, 'heading_recall@5': 0.9690721649484536, 'hs6_recall@5': 0.9123711340206185, 'ahtn8_recall@5': 0.8711340206185567, 'count': 194, 'hierarchy_metric_mode': 'ANCESTOR_PROJECTION_OF_LEAF_RANKING', 'exclusion_annotated_count': 0, 'exclusion_violation_rate': None}


TRAIN best definition branch XVI {'routed_section': 'XVI', 'definition_weight': 0.1, 'k': 100, 'reranker': 'none', 'section_correct': 0.0, 'section_recall@2': 0.0, 'hit@1': 0.6909090909090909, 'mrr': 0.7582575757575757, 'ndcg@10': 0.7864830337082046, 'chapter_recall@5': 0.990909090909091, 'heading_recall@5': 0.9681818181818181, 'hs6_recall@5': 0.8727272727272727, 'ahtn8_recall@5': 0.8545454545454545, 'count': 220, 'hierarchy_metric_mode': 'ANCESTOR_PROJECTION_OF_LEAF_RANKING', 'exclusion_annotated_count': 0, 'exclusion_violation_rate': None}


TRAIN best definition branch XVII {'routed_section': 'XVII', 'definition_weight': 0.0, 'k': 5, 'reranker': 'none', 'section_correct': 0.0, 'section_recall@2': 0.0, 'hit@1': 0.5, 'mrr': 0.5823668714797747, 'ndcg@10': 0.624570772133342, 'chapter_recall@5': 0.989247311827957, 'heading_recall@5': 0.9623655913978495, 'hs6_recall@5': 0.7634408602150538, 'ahtn8_recall@5': 0.6935483870967742, 'count': 186, 'hierarchy_metric_mode': 'ANCESTOR_PROJECTION_OF_LEAF_RANKING', 'exclusion_annotated_count': 0, 'exclusion_violation_rate': None}


,routed_section,definition_weight,k,reranker,section_correct,section_recall@2,hit@1,mrr,ndcg@10,chapter_recall@5,heading_recall@5,hs6_recall@5,ahtn8_recall@5,count,hierarchy_metric_mode,exclusion_annotated_count,exclusion_violation_rate
0,XIV,0.10,5,none,0.0,0.0,0.505155,0.648272,0.717556,1.000000,0.969072,0.912371,0.871134,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
1,XIV,0.05,10,none,0.0,0.0,0.505155,0.646983,0.714350,1.000000,0.963918,0.907216,0.871134,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
2,XIV,0.00,5,none,0.0,0.0,0.505155,0.646467,0.712860,1.000000,0.958763,0.896907,0.871134,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
3,XIV,0.00,10,none,0.0,0.0,0.505155,0.646467,0.712860,1.000000,0.958763,0.896907,0.871134,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
4,XIV,0.00,20,none,0.0,0.0,0.505155,0.646467,0.712860,1.000000,0.958763,0.896907,0.871134,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
5,XIV,0.00,40,none,0.0,0.0,0.505155,0.646467,0.712860,1.000000,0.958763,0.896907,0.871134,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
6,XIV,0.00,60,none,0.0,0.0,0.505155,0.646467,0.712860,1.000000,0.958763,0.896907,0.871134,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
7,XIV,0.00,100,none,0.0,0.0,0.505155,0.646467,0.712860,1.000000,0.958763,0.896907,0.871134,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
8,XIV,0.05,5,none,0.0,0.0,0.505155,0.646467,0.712860,1.000000,0.958763,0.896907,0.871134,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
9,XIV,0.15,5,none,0.0,0.0,0.489691,0.641863,0.713005,1.000000,0.963918,0.912371,0.871134,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None


In [5]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
definition_reranker = CrossEncoder(cfg["reranker_model"], cfg["reranker_revision"], device=device)
retriever._reranker = definition_reranker
print("Evidence cross-encoder:", device, flush=True)

# Include stronger definition branches even when lexical-only selection favors a small weight.
# All candidate pair scores are reusable across pools, blending ratios, and RRF constants.
ENRICH_GRID = [(w, k) for w in (0, 0.05, 0.15, 0.25, 0.5, 0.75, 1.0) for k in (5, 20, 60)]
train_pair_scores = {}
for i, item in enumerate(definition_train_cache):
    case = item["case"]
    key = case["test_id"]
    routed = train_sections[key][0]["section"]
    if routed == "XVII":
        continue
    union = {}
    for weight, k in ENRICH_GRID:
        rows = definition_fusion(train_direct[key], train_definitions[key], weight, k)[:30]
        union.update({r["id"]: r for r in rows})
    passages = retriever.definition_context(list(union.values()))
    with capture_output() as progress:
        scores = definition_reranker.score(case["query"],
            [r.get("rerank_text", r["embedding_text"]) for r in passages])
    train_pair_scores[key] = dict(zip([r["id"] for r in passages], scores, strict=True))
    if i % 20 == 0:
        print("TRAIN source-aware pair scores", i + 1, "/", len(definition_train_cache), flush=True)


def evidence_rank(rows, scores, pool, blend, k):
    ordered = sorted([{**r, "rerank_score": scores[r["id"]]} for r in rows[:pool]],
        key=lambda r: (-r["rerank_score"], r["id"]))
    if blend == 1:
        return ordered
    return unique_codes(fuse({"retrieval": rows[:pool], "rerank": ordered}, k,
        {"retrieval": 1 - blend, "rerank": blend}), pool)

source_trials = []
definition_policies = {}
for section in ("XIV", "XVI", "XVII"):
    subset = [x for x in definition_train_cache if train_sections[x["case"]["test_id"]][0]["section"] == section]
    # Compare to the actual old pipeline, including its selected reranker.
    old_params = baseline_pipeline[section]
    with capture_output() as progress:
        old_rankings = [rerank_candidates(x["case"]["query"], train_direct[x["case"]["test_id"]],
            old_params["reranker"], embedder, definition_reranker,
            old_params.get("rerank_pool", 30), old_params.get("rank_weight", 1), old_params["k"])[:10]
            for x in subset]
    trials = [{"routed_section": section, "definition_weight": 0.0, "k": 20,
        "evidence_rerank": False, "reranker": old_params["reranker"],
        "rerank_pool": old_params.get("rerank_pool", 30),
        "rank_weight": old_params.get("rank_weight", 1),
        **ranking_metrics([x["case"] for x in subset], old_rankings)}]
    trials.extend([{**t, "evidence_rerank": False, "rerank_pool": 30, "rank_weight": 1}
        for t in definition_trials if t["routed_section"] == section])
    if section != "XVII":
        for weight, k in ENRICH_GRID:
            fused = [definition_fusion(train_direct[x["case"]["test_id"]],
                train_definitions[x["case"]["test_id"]], weight, k) for x in subset]
            for pool in (10, 20, 30):
                for blend in (0.25, 0.5, 0.75, 1.0):
                    predictions = [evidence_rank(rows, train_pair_scores[x["case"]["test_id"]],
                        pool, blend, k)[:10] for x, rows in zip(subset, fused, strict=True)]
                    trials.append({"routed_section": section, "definition_weight": weight,
                        "k": k, "evidence_rerank": True, "reranker": "cross_encoder",
                        "rerank_pool": pool, "rank_weight": blend,
                        **ranking_metrics([x["case"] for x in subset], predictions)})
    chosen = max(trials, key=definition_objective)
    definition_policies[section] = {k: chosen[k] for k in (
        "definition_weight", "k", "evidence_rerank", "reranker", "rerank_pool", "rank_weight")}
    source_trials.extend(trials)
    print("TRAIN frozen definition policy", section, chosen, flush=True)
display(pd.DataFrame(source_trials).sort_values(
    ["routed_section", "hit@1", "ahtn8_recall@5", "mrr"], ascending=[True, False, False, False]))
print("TRAIN ONLY policies:", definition_policies)


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 3907.27it/s]

Evidence cross-encoder: mps


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


TRAIN source-aware pair scores 1 / 600


...

TRAIN frozen definition policy XIV {'routed_section': 'XIV', 'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25, 'section_correct': 0.0, 'section_recall@2': 0.0, 'hit@1': 0.5103092783505154, 'mrr': 0.660143593519882, 'ndcg@10': 0.7281441277099436, 'chapter_recall@5': 1.0, 'heading_recall@5': 0.9639175257731959, 'hs6_recall@5': 0.9226804123711341, 'ahtn8_recall@5': 0.8814432989690721, 'count': 194, 'hierarchy_metric_mode': 'ANCESTOR_PROJECTION_OF_LEAF_RANKING', 'exclusion_annotated_count': 0, 'exclusion_violation_rate': None}


TRAIN frozen definition policy XVI {'routed_section': 'XVI', 'definition_weight': 0.5, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 10, 'rank_weight': 0.75, 'section_correct': 0.0, 'section_recall@2': 0.0, 'hit@1': 0.740909090909091, 'mrr': 0.7927272727272726, 'ndcg@10': 0.8106727340091369, 'chapter_recall@5': 0.9818181818181818, 'heading_recall@5': 0.9409090909090909, 'hs6_recall@5': 0.8636363636363636, 'ahtn8_recall@5': 0.8590909090909091, 'count': 220, 'hierarchy_metric_mode': 'ANCESTOR_PROJECTION_OF_LEAF_RANKING', 'exclusion_annotated_count': 0, 'exclusion_violation_rate': None}


TRAIN frozen definition policy XVII {'routed_section': 'XVII', 'definition_weight': 0.0, 'k': 20, 'evidence_rerank': False, 'reranker': 'none', 'rerank_pool': 30, 'rank_weight': 1, 'section_correct': 0.0, 'section_recall@2': 0.0, 'hit@1': 0.5, 'mrr': 0.5823668714797747, 'ndcg@10': 0.624570772133342, 'chapter_recall@5': 0.989247311827957, 'heading_recall@5': 0.9623655913978495, 'hs6_recall@5': 0.7634408602150538, 'ahtn8_recall@5': 0.6935483870967742, 'count': 186, 'hierarchy_metric_mode': 'ANCESTOR_PROJECTION_OF_LEAF_RANKING', 'exclusion_annotated_count': 0, 'exclusion_violation_rate': None}


,routed_section,definition_weight,k,evidence_rerank,reranker,rerank_pool,rank_weight,section_correct,section_recall@2,hit@1,mrr,ndcg@10,chapter_recall@5,heading_recall@5,hs6_recall@5,ahtn8_recall@5,count,hierarchy_metric_mode,exclusion_annotated_count,exclusion_violation_rate
159,XIV,0.00,60,True,cross_encoder,30,0.25,0.0,0.0,0.510309,0.660144,0.728144,1.000000,0.963918,0.922680,0.881443,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
143,XIV,0.00,20,True,cross_encoder,20,0.25,0.0,0.0,0.510309,0.659499,0.727613,1.000000,0.974227,0.932990,0.876289,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
155,XIV,0.00,60,True,cross_encoder,20,0.25,0.0,0.0,0.510309,0.659377,0.727495,1.000000,0.963918,0.927835,0.876289,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
147,XIV,0.00,20,True,cross_encoder,30,0.25,0.0,0.0,0.510309,0.659892,0.727920,1.000000,0.969072,0.927835,0.871134,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
167,XIV,0.05,5,True,cross_encoder,20,0.25,0.0,0.0,0.505155,0.656413,0.727563,1.000000,0.979381,0.927835,0.881443,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
131,XIV,0.00,5,True,cross_encoder,20,0.25,0.0,0.0,0.505155,0.656169,0.727304,1.000000,0.979381,0.927835,0.881443,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
135,XIV,0.00,5,True,cross_encoder,30,0.25,0.0,0.0,0.505155,0.657315,0.729342,1.000000,0.979381,0.927835,0.876289,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
171,XIV,0.05,5,True,cross_encoder,30,0.25,0.0,0.0,0.505155,0.657071,0.728124,1.000000,0.979381,0.938144,0.876289,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
139,XIV,0.00,20,True,cross_encoder,10,0.25,0.0,0.0,0.505155,0.653813,0.718802,1.000000,0.963918,0.922680,0.876289,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
151,XIV,0.00,60,True,cross_encoder,10,0.25,0.0,0.0,0.505155,0.653813,0.718802,1.000000,0.963918,0.922680,0.876289,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None


TRAIN ONLY policies: {'XIV': {'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25}, 'XVI': {'definition_weight': 0.5, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 10, 'rank_weight': 0.75}, 'XVII': {'definition_weight': 0.0, 'k': 20, 'evidence_rerank': False, 'reranker': 'none', 'rerank_pool': 30, 'rank_weight': 1}}


### Chọn trên TRAIN, khóa chính sách, rồi đo diagnostic TEST
Đầu tiên chọn chính sách XIV theo trung bình hai nhóm nguồn tariff/note_example, có điều kiện điểm train tổng không thấp hơn baseline. Sau đó khóa chính sách definition-aware trong selected_retrieval.json, mới chạy so sánh trên 90 sản phẩm test. TEST đã được xem ở các lượt phân tích trước nên điểm này là diagnostic regression, không phải holdout chưa từng được xem.

In [6]:
# The TRAIN XIV set contains 80 tariff-style and 20 Note-based descriptions.
# Prevent the larger source family from masking failures on definition examples.
# Use only TRAIN labels; require overall TRAIN Hit@1 >= the actual old baseline.
subset = [x for x in definition_train_cache
    if train_sections[x["case"]["test_id"]][0]["section"] == "XIV"]
source_balanced_trials = []
for trial in [r for r in source_trials if r["routed_section"] == "XIV"]:
    predicted = []
    for x in subset:
        key = x["case"]["test_id"]
        rows = definition_fusion(train_direct[key], train_definitions[key],
            trial["definition_weight"], trial["k"])
        if trial["evidence_rerank"]:
            rows = evidence_rank(rows, train_pair_scores[key], trial["rerank_pool"],
                trial["rank_weight"], trial["k"])
        predicted.append(rows[:10])
    groups = {}
    for origin in ("tariff", "note_example"):
        indices = [i for i, x in enumerate(subset) if
            ("note_example" if "Note 9/10/11" in x["case"]["evidence_type"] else "tariff") == origin]
        groups[origin] = ranking_metrics([subset[i]["case"] for i in indices],
            [predicted[i] for i in indices])
    source_balanced_trials.append({**trial,
        "macro_source_hit@1": np.mean([g["hit@1"] for g in groups.values()]),
        "macro_source_recall@5": np.mean([g["ahtn8_recall@5"] for g in groups.values()]),
        "macro_source_mrr": np.mean([g["mrr"] for g in groups.values()]),
        "tariff_hit@1": groups["tariff"]["hit@1"],
        "note_example_hit@1": groups["note_example"]["hit@1"],
        "tariff_recall@5": groups["tariff"]["ahtn8_recall@5"],
        "note_example_recall@5": groups["note_example"]["ahtn8_recall@5"]})
baseline_train_hit = next(r["hit@1"] for r in source_trials if r["routed_section"] == "XIV")
eligible = [r for r in source_balanced_trials if r["hit@1"] >= baseline_train_hit]
balanced_chosen = max(eligible, key=lambda r: (r["macro_source_hit@1"],
    r["macro_source_recall@5"], r["macro_source_mrr"], r["hit@1"]))
definition_policies["XIV"] = {k: balanced_chosen[k] for k in (
    "definition_weight", "k", "evidence_rerank", "reranker", "rerank_pool", "rank_weight")}
display(pd.DataFrame(source_balanced_trials).sort_values(
    ["macro_source_hit@1", "macro_source_recall@5", "macro_source_mrr"],
    ascending=False).head(20))
print("TRAIN source-balanced XIV policy:", balanced_chosen)
print("Selection uses only TRAIN, equal source-family weights, and an overall TRAIN baseline guard.")


,routed_section,definition_weight,k,evidence_rerank,reranker,rerank_pool,rank_weight,section_correct,section_recall@2,hit@1,mrr,ndcg@10,chapter_recall@5,heading_recall@5,hs6_recall@5,ahtn8_recall@5,count,hierarchy_metric_mode,exclusion_annotated_count,exclusion_violation_rate,macro_source_hit@1,macro_source_recall@5,macro_source_mrr,tariff_hit@1,note_example_hit@1,tariff_recall@5,note_example_recall@5
145,XIV,0.00,20,True,cross_encoder,20,0.75,0.0,0.0,0.489691,0.664059,0.734665,1.0,0.969072,0.917526,0.902062,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,0.385178,0.846531,0.575396,0.554140,0.216216,0.936306,0.756757
133,XIV,0.00,5,True,cross_encoder,20,0.75,0.0,0.0,0.489691,0.664903,0.736429,1.0,0.969072,0.917526,0.896907,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,0.385178,0.843347,0.578815,0.554140,0.216216,0.929936,0.756757
169,XIV,0.05,5,True,cross_encoder,20,0.75,0.0,0.0,0.489691,0.664646,0.736203,1.0,0.969072,0.917526,0.896907,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,0.385178,0.843347,0.578140,0.554140,0.216216,0.929936,0.756757
134,XIV,0.00,5,True,cross_encoder,20,1.00,0.0,0.0,0.489691,0.665863,0.735714,1.0,0.963918,0.922680,0.881443,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,0.385178,0.833792,0.590524,0.554140,0.216216,0.910828,0.756757
146,XIV,0.00,20,True,cross_encoder,20,1.00,0.0,0.0,0.489691,0.665863,0.735714,1.0,0.963918,0.922680,0.881443,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,0.385178,0.833792,0.590524,0.554140,0.216216,0.910828,0.756757
158,XIV,0.00,60,True,cross_encoder,20,1.00,0.0,0.0,0.489691,0.665863,0.735714,1.0,0.963918,0.922680,0.881443,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,0.385178,0.833792,0.590524,0.554140,0.216216,0.910828,0.756757
170,XIV,0.05,5,True,cross_encoder,20,1.00,0.0,0.0,0.489691,0.665863,0.735714,1.0,0.963918,0.922680,0.881443,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,0.385178,0.833792,0.590524,0.554140,0.216216,0.910828,0.756757
182,XIV,0.05,20,True,cross_encoder,20,1.00,0.0,0.0,0.489691,0.665691,0.735556,1.0,0.974227,0.922680,0.876289,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,0.385178,0.820279,0.590073,0.554140,0.216216,0.910828,0.729730
181,XIV,0.05,20,True,cross_encoder,20,0.75,0.0,0.0,0.484536,0.661346,0.732655,1.0,0.979381,0.917526,0.902062,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,0.381993,0.846531,0.573720,0.547771,0.216216,0.936306,0.756757
159,XIV,0.00,60,True,cross_encoder,30,0.25,0.0,0.0,0.510309,0.660144,0.728144,1.0,0.963918,0.922680,0.881443,194,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,0.377259,0.771820,0.541945,0.592357,0.162162,0.949045,0.594595


TRAIN source-balanced XIV policy: {'routed_section': 'XIV', 'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25, 'section_correct': 0.0, 'section_recall@2': 0.0, 'hit@1': 0.5103092783505154, 'mrr': 0.660143593519882, 'ndcg@10': 0.7281441277099436, 'chapter_recall@5': 1.0, 'heading_recall@5': 0.9639175257731959, 'hs6_recall@5': 0.9226804123711341, 'ahtn8_recall@5': 0.8814432989690721, 'count': 194, 'hierarchy_metric_mode': 'ANCESTOR_PROJECTION_OF_LEAF_RANKING', 'exclusion_annotated_count': 0, 'exclusion_violation_rate': None, 'macro_source_hit@1': np.float64(0.37725942503012566), 'macro_source_recall@5': np.float64(0.771819590290928), 'macro_source_mrr': np.float64(0.5419452778529212), 'tariff_hit@1': 0.5923566878980892, 'note_example_hit@1': 0.16216216216216217, 'tariff_recall@5': 0.9490445859872612, 'note_example_recall@5': 0.5945945945945946}
Selection uses only TRAIN, equal source-family weights, and an overal

In [7]:
definition_selection = {**baseline_selection,
    "definition_policies": definition_policies,
    "definition_version": "positive-definition-graph-bm25-v1",
    "definition_train_trials": len(source_trials),
    "definition_source_hash": digest({k: v["text"] for k, v in views.items()}),
    "baseline_pipeline_policies": baseline_pipeline,
    "baseline_selection_scope": baseline_selection.get("baseline_selection_scope", baseline_selection["selection_scope"]),
    "selection_scope": "TRAIN_ONLY_SELECTION; EXISTING_TEST_ALREADY_INSPECTED_FOR_DIAGNOSIS",
    "evaluation_status": "TRAIN_SELECTED; EXISTING_TEST_IS_DIAGNOSTIC_AFTER_FAILURE_INSPECTION"}
write_json(ROOT / "config/selected_retrieval.json", definition_selection)
definition_frozen_hash = digest(definition_selection)
print("Definition policy frozen before diagnostic TEST:", definition_frozen_hash)
with capture_output() as progress:
    definition_test_cache = branch_cache(retriever, test_cases, filters={"levels": [8]}, pool=100)
    definition_router_test_cache = branch_cache(retriever, test_cases,
        filters={"sections": ["XIV", "XVI", "XVII"]}, pool=100)
test_sections_new = {x["case"]["test_id"]: retriever.section_candidates(
    fuse(x["branches"], router_policy["k"], router_policy["weights"]))
    for x in definition_router_test_cache}
before_rankings, after_rankings = [], []
case_diagnosis = []
for i, item in enumerate(definition_test_cache):
    case = item["case"]
    query = case["query"]
    sections = test_sections_new[case["test_id"]]
    routed = sections[0]["section"]
    old, new = baseline_pipeline[routed], definition_policies[routed]
    direct = candidate_paths(item["branches"], sections, section_policies, old)["candidates"]
    with capture_output() as progress:
        before = rerank_candidates(query, direct, old["reranker"], embedder,
            definition_reranker, old.get("rerank_pool", 30), old.get("rank_weight", 1), old["k"])[:10]
        definitions = retriever.definition_candidates(query, 100, [s["section"] for s in sections])
        fused = definition_fusion(direct, definitions, new["definition_weight"], new["k"])
        contextual = retriever.definition_context(fused) if new["evidence_rerank"] else fused
        after = rerank_candidates(query, contextual, new["reranker"], embedder,
            definition_reranker, new["rerank_pool"], new["rank_weight"], new["k"])[:10]
    before_rankings.append(before)
    after_rankings.append(after)
    def gold_rank(rows):
        codes = [r["code"] for r in rows]
        return codes.index(case["expected_hs_8"]) + 1 if case["expected_hs_8"] in codes else None
    gold_view = next((v for v in views.values() if v["record"]["code"] == case["expected_hs_8"]), None)
    case_diagnosis.append({"test_id": case["test_id"], "section": case["section"],
        "variant": case["variant"], "query": query, "gold": case["expected_hs_8"],
        "routed_section": routed, "section_candidates": [s["section"] for s in sections],
        "before_rank": gold_rank(before), "after_rank": gold_rank(after),
        "before_top5": [r["code"] for r in before[:5]],
        "after_top5": [r["code"] for r in after[:5]],
        "gold_definition_paths": gold_view["paths"] if gold_view else [],
        "gold_rank_in_definition_branch": gold_rank(definitions),
        "ranking_policy": new})
    if i % 20 == 0:
        print("Diagnostic TEST", i + 1, "/", len(definition_test_cache), flush=True)
assert digest(read(ROOT / "config/selected_retrieval.json")) == definition_frozen_hash
comparison = []
for section in ("XIV", "XVI", "XVII"):
    for variant in ("name_only", "name_and_spec"):
        indices = [i for i, x in enumerate(definition_test_cache)
            if x["case"]["section"] == section and x["case"]["variant"] == variant]
        selected_cases = [definition_test_cache[i]["case"] for i in indices]
        for label, rankings in [("baseline", before_rankings), ("definition_aware", after_rankings)]:
            metric = ranking_metrics(selected_cases, [rankings[i] for i in indices],
                [test_sections_new[selected_cases[j]["test_id"]] for j in range(len(indices))])
            comparison.append({"section": section, "variant": variant, "pipeline": label, **metric})
display(pd.DataFrame(comparison))
display(pd.DataFrame(case_diagnosis))
print("Diagnostic only; no independent new holdout or legal-classification accuracy claim.")


Definition policy frozen before diagnostic TEST: 2407c8f2e48154f19b9f17b5b3677b449d1d6b4c9ed94950bd5c3e560b72b5f0


Diagnostic TEST 1 / 180


Diagnostic TEST 21 / 180


Diagnostic TEST 41 / 180


Diagnostic TEST 61 / 180


...

Diagnostic TEST 141 / 180


Diagnostic TEST 161 / 180


,section,variant,pipeline,section_correct,section_recall@2,hit@1,mrr,ndcg@10,chapter_recall@5,heading_recall@5,hs6_recall@5,ahtn8_recall@5,count,hierarchy_metric_mode,exclusion_annotated_count,exclusion_violation_rate
0,XIV,name_only,baseline,0.966667,1.0,0.066667,0.189206,0.256825,1.000000,0.566667,0.500000,0.433333,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
1,XIV,name_only,definition_aware,0.966667,1.0,0.066667,0.220132,0.295927,1.000000,0.600000,0.500000,0.433333,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
2,XIV,name_and_spec,baseline,0.933333,1.0,0.033333,0.208095,0.287518,1.000000,0.500000,0.466667,0.433333,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
3,XIV,name_and_spec,definition_aware,0.933333,1.0,0.066667,0.259722,0.357390,1.000000,0.633333,0.566667,0.500000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
4,XVI,name_only,baseline,1.000000,1.0,0.900000,0.944444,0.958729,1.000000,1.000000,1.000000,1.000000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
5,XVI,name_only,definition_aware,1.000000,1.0,0.900000,0.916667,0.921031,0.933333,0.933333,0.933333,0.933333,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
6,XVI,name_and_spec,baseline,1.000000,1.0,0.966667,0.983333,0.987698,1.000000,1.000000,1.000000,1.000000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
7,XVI,name_and_spec,definition_aware,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
8,XVII,name_only,baseline,0.966667,1.0,0.333333,0.478466,0.555571,1.000000,0.933333,0.800000,0.700000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
9,XVII,name_only,definition_aware,0.966667,1.0,0.300000,0.461799,0.543269,1.000000,0.933333,0.800000,0.700000,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None


,test_id,section,variant,query,gold,routed_section,section_candidates,before_rank,after_rank,before_top5,after_top5,gold_definition_paths,gold_rank_in_definition_branch,ranking_policy
0,XIV_TE001:name_only,XIV,name_only,Hoa tai bằng bạc,71131190,XIV,"[XIV, XVI]",NaN,NaN,"[71162000, 85064000, 71141100, 85393190, 71159020]","[71141100, 71162000, 85064000, 71159020, 71110010]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]",5.0,"{'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25}"
1,XIV_TE001:name_and_spec,XIV,name_and_spec,Hoa tai bằng bạc. Hoa tai bằng bạc,71131190,XIV,"[XIV, XVI]",10.0,6.0,"[71162000, 71141100, 85064000, 71159020, 84283390]","[71141100, 85064000, 71162000, 71159020, 71159010]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]",5.0,"{'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25}"
2,XIV_TE002:name_only,XIV,name_only,Dây đồng hồ bằng bạc,71131190,XVI,"[XVI, XIV]",4.0,2.0,"[85064000, 71141100, 71159010, 71131190, 71110010]","[71131110, 71131190, 71141100, 71131910, 71132010]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]",2.0,"{'definition_weight': 0.5, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 10, 'rank_weight': 0.75}"
3,XIV_TE002:name_and_spec,XIV,name_and_spec,Dây đồng hồ bằng bạc. Dây đồng hồ bằng bạc,71131190,XIV,"[XIV, XVI]",NaN,NaN,"[71141100, 71142000, 85064000, 71159010, 85441190]","[71141100, 85064000, 71142000, 71159010, 71141900]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]",2.0,"{'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25}"
4,XIV_TE003:name_only,XIV,name_only,Vật trang trí đồng hồ bỏ túi (fob) bằng bạc,71131190,XIV,"[XIV, XVI]",5.0,3.0,"[71179094, 71131110, 71141100, 71142000, 71131190]","[71131110, 71179094, 71131190, 71141100, 71142000]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]",5.0,"{'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25}"
5,XIV_TE003:name_and_spec,XIV,name_and_spec,Vật trang trí đồng hồ bỏ túi (fob) bằng bạc. Vật trang trí đồng hồ bỏ túi (fob) bằng bạc,71131190,XIV,"[XIV, XVI]",4.0,2.0,"[71131110, 71179094, 71141100, 71131190, 71142000]","[71131110, 71131190, 71179094, 71141100, 71142000]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]",5.0,"{'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25}"
6,XIV_TE004:name_only,XIV,name_only,Mặt dây chuyền bằng bạc,71131190,XIV,"[XIV, XVI]",NaN,NaN,"[71141100, 84623210, 71159020, 84623290, 71159010]","[71141100, 71159010, 71159020, 84623210, 85064000]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]",2.0,"{'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25}"
7,XIV_TE004:name_and_spec,XIV,name_and_spec,Mặt dây chuyền bằng bạc. Mặt dây chuyền bằng bạc,71131190,XIV,"[XIV, XVI]",NaN,NaN,"[71141100, 84623210, 71159020, 84623290, 71159010]","[71141100, 71159010, 71159020, 84623210, 85064000]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]",2.0,"{'definition_weight': 0, 'k'

Diagnostic only; no independent new holdout or legal-classification accuracy claim.


### Runtime parity, source-family analysis, latency and validation
Validate the live routed_search path against the policy already frozen after TRAIN source-family selection, including boundary cases and the pre-truncation Top-2 router fix. Keep completed outputs. Separate candidate omission from ranking losses and retrieval from legal reasoning.

In [8]:
import importlib
import subprocess

import rag.retrieval as retrieval_module

# The source-balanced TRAIN objective was selected before the final freeze and evaluation.
assert definition_policies == read(ROOT / "config/selected_retrieval.json")["definition_policies"]
selection = read(ROOT / "config/selected_retrieval.json")
selection["definition_objective"] = (
    "XIV macro source-family Hit@1 -> Recall@5 -> MRR; overall TRAIN Hit@1 >= old baseline; "
    "other Sections retain TRAIN overall objective")
write_json(ROOT / "config/selected_retrieval.json", selection)
final_policy_hash = digest(selection)
importlib.reload(retrieval_module)
runtime = retrieval_module.Retriever(conn, embedder)
runtime._reranker = definition_reranker

# Source-family diagnostic on TRAIN, independent of TEST metrics.
train_origin_rows = []
for section in ("XIV", "XVI", "XVII"):
    subset = [x for x in definition_train_cache if x["case"]["section"] == section]
    for origin in ("tariff", "note_example"):
        selected_items = [x for x in subset if
            ("note_example" if "Note 9/10/11" in x["case"]["evidence_type"] else "tariff") == origin]
        if not selected_items:
            continue
        for label in ("baseline", "definition_aware"):
            predictions = []
            for x in selected_items:
                key = x["case"]["test_id"]
                routed = train_sections[key][0]["section"]
                old = baseline_pipeline[routed]
                rows = train_direct[key]
                if label == "baseline":
                    with capture_output():
                        rows = rerank_candidates(x["case"]["query"], rows, old["reranker"], embedder,
                            definition_reranker, old.get("rerank_pool", 30), old.get("rank_weight", 1), old["k"])
                else:
                    policy = definition_policies[routed]
                    rows = definition_fusion(rows, train_definitions[key], policy["definition_weight"], policy["k"])
                    if policy["evidence_rerank"]:
                        rows = evidence_rank(rows, train_pair_scores[key], policy["rerank_pool"],
                            policy["rank_weight"], policy["k"])
                predictions.append(rows[:10])
            train_origin_rows.append({"section": section, "source_family": origin, "pipeline": label,
                **ranking_metrics([x["case"] for x in selected_items], predictions)})
display(pd.DataFrame(train_origin_rows))

# Inspect runtime parity for all three Sections, plus misrouting and truncation cases.
parity_ids = {"XIV_TE001:name_and_spec", "XIV_TE002:name_and_spec", "XIV_TE007:name_and_spec",
    "XIV_TE019:name_and_spec", "XVI_TE001:name_and_spec", "XVI_TE014:name_only",
    "XVII_TE001:name_and_spec", "XVII_TE015:name_and_spec"}
parity_rows = []
for i, item in enumerate(definition_test_cache):
    case = item["case"]
    if case["test_id"] not in parity_ids:
        continue
    with capture_output():
        result = runtime.routed_search(case["query"])
    actual = [r["code"] for r in result["candidates"]]
    expected = [r["code"] for r in after_rankings[i]]
    assert actual == expected, (case["test_id"], actual, expected)
    expected_sections = test_sections_new[case["test_id"]]
    assert [s["section"] for s in result["section_candidates"]] == [s["section"] for s in expected_sections]
    np.testing.assert_allclose([s["score"] for s in result["section_candidates"]],
        [s["score"] for s in expected_sections], rtol=1e-12, atol=1e-15)
    assert result["validation_status"] == "REQUIRES_GIR_EXCLUSION_REASONING"
    parity_rows.append({"test_id": case["test_id"], "runtime_equals_benchmark": True,
        "top5": actual[:5], "definition_policy": result["definition_policy"],
        "definition_source_ids": [x["id"] for x in result["definition_evidence"]]})
display(pd.DataFrame(parity_rows))

# Show the concrete remaining candidate/ranking losses without changing the policy.
failure_frame = pd.DataFrame(case_diagnosis)
failure_frame = failure_frame[(failure_frame["section"] == "XIV") &
    (failure_frame["variant"] == "name_and_spec") &
    (failure_frame["after_rank"].isna() | (failure_frame["after_rank"] > 1))]
display(failure_frame[["test_id", "query", "gold", "before_rank", "after_rank",
    "gold_rank_in_definition_branch", "after_top5", "gold_definition_paths"]])

validation_env = {**os.environ, "RAG_RUN_DB_TESTS": "1", "TOKENIZERS_PARALLELISM": "false"}
for command in [
    [str(ROOT / ".venv/bin/ruff"), "check", "rag", "tests", "notebooks"],
    [str(ROOT / ".venv/bin/mypy"), "rag"],
    [str(ROOT / ".venv/bin/bandit"), "-r", "rag", "-ll", "-q"],
    [str(ROOT / ".venv/bin/python"), "-m", "pytest", "tests", "-q"],
]:
    result = subprocess.run(command, cwd=ROOT, env=validation_env, capture_output=True, text=True)
    print(" ".join(command), "exit", result.returncode)
    print(result.stdout, result.stderr)
    assert result.returncode == 0
assert digest(read(ROOT / "config/selected_retrieval.json")) == final_policy_hash
print("Runtime parity samples:", len(parity_rows), "Frozen selection hash:", final_policy_hash)


,section,source_family,pipeline,section_correct,section_recall@2,hit@1,mrr,ndcg@10,chapter_recall@5,heading_recall@5,hs6_recall@5,ahtn8_recall@5,count,hierarchy_metric_mode,exclusion_annotated_count,exclusion_violation_rate
0,XIV,tariff,baseline,0.0,0.0,0.57500,0.706473,0.767730,0.99375,0.98125,0.94375,0.91875,160,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
1,XIV,tariff,definition_aware,0.0,0.0,0.58125,0.719288,0.780612,0.99375,0.98750,0.95000,0.93125,160,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
2,XIV,note_example,baseline,0.0,0.0,0.15000,0.309474,0.386453,0.97500,0.77500,0.60000,0.55000,40,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
3,XIV,note_example,definition_aware,0.0,0.0,0.15000,0.354544,0.450267,1.00000,0.85000,0.75000,0.62500,40,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
4,XVI,tariff,baseline,0.0,0.0,0.79500,0.854631,0.875936,0.99500,0.99000,0.94500,0.92000,200,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
5,XVI,tariff,definition_aware,0.0,0.0,0.81500,0.864583,0.881190,0.99000,0.96500,0.93000,0.92000,200,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
6,XVII,tariff,baseline,0.0,0.0,0.46500,0.543726,0.585808,0.99500,0.96000,0.72500,0.65000,200,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None
7,XVII,tariff,definition_aware,0.0,0.0,0.46500,0.543018,0.583157,0.98500,0.94000,0.71500,0.65500,200,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None


,test_id,runtime_equals_benchmark,top5,definition_policy,definition_source_ids
0,XIV_TE001:name_and_spec,True,"[71141100, 85064000, 71162000, 71159020, 71159010]","{'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25}","[tt31:XIV_CH71_NOTE_10, tt31:XIV_CH71_NOTE_09]"
1,XIV_TE002:name_and_spec,True,"[71141100, 85064000, 71142000, 71159010, 71141900]","{'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25}","[tt31:XIV_CH71_NOTE_10, tt31:XIV_CH71_NOTE_09]"
2,XIV_TE007:name_and_spec,True,"[71131110, 71131190, 71141100, 71131990, 71141900]","{'definition_weight': 0.5, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 10, 'rank_weight': 0.75}","[tt31:XIV_CH71_NOTE_09, tt31:XIV_CH71_NOTE_10, tt31:XIV_CH71_SUBHEADING_NOTE_01]"
3,XIV_TE019:name_and_spec,True,"[71070000, 71132010, 71132090, 71142000, 71090000]","{'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25}","[tt31:XIV_CH71_NOTE_09, tt31:XIV_CH71_NOTE_10]"
4,XVI_TE001:name_and_spec,True,"[85101000, 85109000, 85103000, 85102000, 85094000]","{'definition_weight': 0.5, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 10, 'rank_weight': 0.75}",[tt31:CH88_SUBNOTE_1]
5,XVI_TE014:name_only,True,"[85311030, 85311020, 85311090, 85311010, 85312000]","{'definition_weight': 0.5, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 10, 'rank_weight': 0.75}",[tt31:CH88_SUBNOTE_1]
6,XVII_TE001:name_and_spec,True,"[87042121, 87045272, 87042111, 87042247, 87044121]","{'definition_weight': 0.0, 'k': 20, 'evidence_rerank': False, 'reranker': 'none', 'rerank_pool': 30, 'rank_weight': 1}",[]
7,XVII_TE015:name_and_spec,True,"[87120020, 87150000, 87149610, 87149510, 87089930]","{'definition_weight': 0.0, 'k': 20, 'evidence_rerank': False, 'reranker': 'none', 'rerank_pool': 30, 'rank_weight': 1}",[tt31:CH87_NOTE_4]


,test_id,query,gold,before_rank,after_rank,gold_rank_in_definition_branch,after_top5,gold_definition_paths
1,XIV_TE001:name_and_spec,Hoa tai bằng bạc. Hoa tai bằng bạc,71131190,10.0,6.0,5.0,"[71141100, 85064000, 71162000, 71159020, 71159010]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]"
3,XIV_TE002:name_and_spec,Dây đồng hồ bằng bạc. Dây đồng hồ bằng bạc,71131190,NaN,NaN,2.0,"[71141100, 85064000, 71142000, 71159010, 71141900]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]"
5,XIV_TE003:name_and_spec,Vật trang trí đồng hồ bỏ túi (fob) bằng bạc. Vật trang trí đồng hồ bỏ túi (fob) bằng bạc,71131190,4.0,2.0,5.0,"[71131110, 71131190, 71179094, 71141100, 71142000]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]"
7,XIV_TE004:name_and_spec,Mặt dây chuyền bằng bạc. Mặt dây chuyền bằng bạc,71131190,NaN,NaN,2.0,"[71141100, 71159010, 71159020, 84623210, 85064000]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]"
9,XIV_TE005:name_and_spec,Khuy măng sét bằng bạc. Khuy măng sét bằng bạc,71131190,NaN,NaN,7.0,"[71171110, 71171190, 71142000, 71141100, 71181010]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]"
11,XIV_TE006:name_and_spec,Khuy rời trang trí áo bằng bạc. Khuy rời trang trí áo bằng bạc,71131190,2.0,3.0,7.0,"[71131110, 71141900, 71131190, 71141100, 71171190]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]"
13,XIV_TE007:name_and_spec,Hộp đựng thuốc lá điếu bằng bạc. Hộp đựng thuốc lá điếu bằng bạc,71131190,NaN,2.0,2.0,"[71131110, 71131190, 71141100, 71131990, 71141900]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]"
15,XIV_TE008:name_and_spec,Hộp đựng thuốc viên bằng bạc. Hộp đựng thuốc viên bằng bạc,71131190,NaN,NaN,2.0,"[71141100, 85064000, 71141900, 84801000, 71189020]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]"
17,XIV_TE009:name_and_spec,Hộp phấn bằng bạc. Hộp phấn bằng bạc,71131190,NaN,8.0,5.0,"[71131110, 71141100, 85064000, 71142000, 71141900]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131190'}]"
21,XIV_TE011:name_and_spec,Nhẫn bằng vàng. Nhẫn bằng vàng,71131990,NaN,NaN,12.0,"[71159010, 71141900, 71142000, 71141100, 71159020]","[{'note_id': 'tt31:XIV_CH71_NOTE_09', 'relation': 'DEFINES_FOR', 'target_id': 'tt31:HS4:7113', 'candidate_id': 'tt31:HS8:71131990'}]"


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/ruff check rag tests notebooks exit 0
All checks passed!
 


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/mypy rag exit 0
Success: no issues found in 6 source files
 


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/bandit -r rag -ll -q exit 0
 


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/python -m pytest tests -q exit 0
.....................................................                    [100%]
53 passed in 23.28s
 
Runtime parity samples: 8 Frozen selection hash: 9dd53b5505193553ec811c05edd1311fb3860e28c792ddc5594593237e33d52f


In [9]:
from rag.db import counts

# Uncached query/reranker timing; corpus indexes and model weights are warm.
# Same candidate-stage scope as the earlier latency table, excluding Notes/GIR/evidence expansion.
new_latency_rows = []
for section in ("XIV", "XVI", "XVII"):
    samples = [x["case"] for x in definition_test_cache
        if x["case"]["section"] == section and x["case"]["variant"] == "name_and_spec"][:3]
    for case in samples:
        start = time.perf_counter()
        with capture_output():
            cross_item = branch_cache(runtime, [case],
                filters={"sections": ["XIV", "XVI", "XVII"]}, pool=100, use_cache=False)[0]
            sections = runtime.section_candidates(fuse(cross_item["branches"],
                router_policy["k"], router_policy["weights"]))
            leaf_item = branch_cache(runtime, [case], filters={"levels": [8]},
                pool=100, use_cache=False)[0]
            old = baseline_pipeline[sections[0]["section"]]
            policy = definition_policies[sections[0]["section"]]
            candidates = candidate_paths(leaf_item["branches"], sections,
                section_policies, old)["candidates"]
            definitions = runtime.definition_candidates(case["query"], 100,
                [s["section"] for s in sections])
            candidates = definition_fusion(candidates, definitions,
                policy["definition_weight"], policy["k"])
            if policy["evidence_rerank"]:
                candidates = runtime.definition_context(candidates)
            ranked = rerank_candidates(case["query"], candidates, policy["reranker"],
                embedder, definition_reranker, policy["rerank_pool"], policy["rank_weight"],
                policy["k"], use_cache=False)
        new_latency_rows.append({"section": section, "test_id": case["test_id"],
            "uncached_candidate_ms": (time.perf_counter() - start) * 1000,
            "top5": [r["code"] for r in ranked[:5]]})
display(pd.DataFrame(new_latency_rows))
display(pd.DataFrame(new_latency_rows).groupby("section").agg(
    samples=("uncached_candidate_ms", "count"),
    p50_ms=("uncached_candidate_ms", "median"),
    p95_ms=("uncached_candidate_ms", lambda x: np.percentile(x, 95))))
print("Local 3 samples/Section; indexes/model warm, query/reranker caches bypassed; not a production SLO.")

display(pd.DataFrame([counts(conn, runtime.snapshot, runtime.profile)]))
print("Selected source-aware policies:", definition_policies)
print("Candidate retrieval improvement; GIR/exclusion reasoning and final HS conclusions remain separate.")


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


,section,test_id,uncached_candidate_ms,top5
0,XIV,XIV_TE001:name_and_spec,3450.297500,"[71141100, 85064000, 71162000, 71159020, 71159010]"
1,XIV,XIV_TE002:name_and_spec,1569.872750,"[71141100, 85064000, 71142000, 71159010, 71141900]"
2,XIV,XIV_TE003:name_and_spec,1698.466041,"[71131110, 71131190, 71179094, 71141100, 71142000]"
3,XVI,XVI_TE001:name_and_spec,681.455250,"[85101000, 85109000, 85103000, 85102000, 85094000]"
4,XVI,XVI_TE002:name_and_spec,720.003958,"[85102000, 85109000, 85101000, 85103000, 84672900]"
5,XVI,XVI_TE003:name_and_spec,814.543917,"[85103000, 85109000, 85101000, 85102000, 88022090]"
6,XVII,XVII_TE001:name_and_spec,921.086500,"[87042121, 87045272, 87042111, 87042247, 87044121]"
7,XVII,XVII_TE002:name_and_spec,617.851917,"[87042122, 87042222, 87045262, 87043122, 87045282]"
8,XVII,XVII_TE003:name_and_spec,652.493542,"[87042123, 87045263, 87042223, 87045283, 87045243]"


,samples,p50_ms,p95_ms
section,,,
XIV,3,1698.466041,3275.114354
XVI,3,720.003958,805.089921
XVII,3,652.493542,894.227204


Local 3 samples/Section; indexes/model warm, query/reranker caches bypassed; not a production SLO.


,chunks,embeddings,nodes,edges
0,5356,5356,5521,6655


Selected source-aware policies: {'XIV': {'definition_weight': 0, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.25}, 'XVI': {'definition_weight': 0.5, 'k': 60, 'evidence_rerank': True, 'reranker': 'cross_encoder', 'rerank_pool': 10, 'rank_weight': 0.75}, 'XVII': {'definition_weight': 0.0, 'k': 20, 'evidence_rerank': False, 'reranker': 'none', 'rerank_pool': 30, 'rank_weight': 1}}
Candidate retrieval improvement; GIR/exclusion reasoning and final HS conclusions remain separate.


In [10]:
# Final verification; independent of earlier notebook kernel variables.
import os
import subprocess

from IPython.utils.io import capture_output

from rag.data import ROOT, config, digest, load_corpus, read
from rag.db import connect
from rag.embedding import Embedder
from rag.evaluation import QA_CHUNK_TYPES, branch_cache, discover_qa
from rag.retrieval import Retriever, fuse, rerank_evidence

selection_before_checks = digest(read(ROOT / 'config/selected_retrieval.json'))
selection_for_checks = read(ROOT / 'config/selected_retrieval.json')
if selection_for_checks.get('qa'):
    parity_cases = [c for c in discover_qa(load_corpus()) if c['split']=='train'][::20]
    with connect() as parity_conn:
        with capture_output():
            parity_embedder = Embedder(config())
        parity_runtime = Retriever(parity_conn, parity_embedder)
        parity_policy = parity_runtime.qa_policy
        parity_cache = branch_cache(parity_runtime, parity_cases, intent='knowledge', pool=100,
            filters={'chunk_types': QA_CHUNK_TYPES}, mask_questioned_codes=True)
        for item in parity_cache:
            with capture_output():
                live = parity_runtime.qa_search(item['case']['query'])
            cached = fuse(item['branches'], parity_policy['k'], parity_policy['weights'])
            if parity_policy.get('reranker') == 'cross_encoder':
                cached = rerank_evidence(item['case']['query'], cached, parity_runtime._reranker,
                    parity_policy['rerank_pool'], parity_policy['rank_weight'], parity_policy['k'])
            assert [r['id'] for r in cached[:10]] == [r['id'] for r in live['matches']]
        print('QA runtime versus train-cache benchmark: 10/10 exact Top-10 parity.')

validation_env = {**os.environ, 'RAG_RUN_DB_TESTS': '1', 'TOKENIZERS_PARALLELISM': 'false'}
for command in [
    [str(ROOT / '.venv/bin/ruff'), 'check', 'rag', 'tests', 'notebooks'],
    [str(ROOT / '.venv/bin/mypy'), 'rag'],
    [str(ROOT / '.venv/bin/bandit'), '-r', 'rag', '-ll', '-q'],
    [str(ROOT / '.venv/bin/python'), '-m', 'pytest', 'tests', '-q'],
    ['git', 'diff', '--check'],
]:
    checked = subprocess.run(command, cwd=ROOT, env=validation_env, capture_output=True, text=True, check=False)
    print(' '.join(command), 'exit', checked.returncode)
    print(checked.stdout or checked.stderr or 'PASS')
    assert checked.returncode == 0
assert digest(read(ROOT / 'config/selected_retrieval.json')) == selection_before_checks
print('Final checks passed. Frozen policy unchanged:', selection_before_checks)


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/ruff check rag tests notebooks exit 0
All checks passed!



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/mypy rag exit 0
Success: no issues found in 6 source files



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/bandit -r rag -ll -q exit 0
PASS


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/python -m pytest tests -q exit 0
.....................................................                    [100%]
53 passed in 22.83s

git diff --check exit 0
PASS
Final checks passed. Frozen policy unchanged: 9dd53b5505193553ec811c05edd1311fb3860e28c792ddc5594593237e33d52f


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## QA: lịch sử benchmark 200/50

Các output QA bên dưới thuộc snapshot và bộ dữ liệu cũ; phần QA của notebook 01 đánh giá bộ QA 320/150 mới. Không dùng kết quả lịch sử để chọn policy hiện tại.


In [1]:
# QA policy selection: only supplied expanded_200 labels enter this cell.
from collections import Counter

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from IPython.utils.io import capture_output

from rag.data import ROOT, config, digest, load_corpus, read, write_json
from rag.db import connect
from rag.embedding import CrossEncoder, Embedder
from rag.evaluation import (
    CONFIGURATIONS,
    QA_CHUNK_TYPES,
    branch_cache,
    discover_qa,
    qa_direct_coverage,
    qa_objective,
    sweep_qa_weights,
)
from rag.retrieval import Retriever, fuse, rerank_evidence

qa_bundle = load_corpus()
qa_all = discover_qa(qa_bundle)
qa_train = [c for c in qa_all if c['split'] == 'train']
qa_test = [c for c in qa_all if c['split'] == 'test']
assert (len(qa_train), len(qa_test)) == (200, 50)
qa_conn = connect()
with capture_output():
    qa_embedder = Embedder(config())
qa_runtime = Retriever(qa_conn, qa_embedder)
assert qa_runtime.snapshot == qa_bundle['snapshot']
hs_before_qa = read(ROOT / 'config/selected_retrieval.json')
qa_cache = branch_cache(qa_runtime, qa_train, intent='knowledge', pool=100,
    filters={'chunk_types': QA_CHUNK_TYPES}, mask_questioned_codes=True)
print('Train branches cached. Sweeping weights and RRF k on 200 queries only.', flush=True)
qa_trials = []
qa_family_best = []
for family in CONFIGURATIONS:
    runs = sweep_qa_weights(qa_cache, family)
    qa_trials.extend(runs)
    qa_family_best.append(runs[0])
    print(family, 'trials', len(runs), 'best', runs[0], flush=True)
display(pd.DataFrame(qa_family_best))
qa_base = max(qa_trials, key=qa_objective)
print('Base trials:', len(qa_trials), 'best:', qa_base, flush=True)

# Official cross-encoder: score source chunks by ID, never deduplicate legal rows by HS code.
with capture_output():
    qa_reranker = CrossEncoder(config()['reranker_model'], config()['reranker_revision'],
        device='mps' if torch.backends.mps.is_available() else 'cpu')
qa_advanced = []
base_candidates = [qa_base]
full_base = next(r for r in qa_family_best if r['configuration'] == 'C10')
if full_base != qa_base:
    base_candidates.append(full_base)
for base in base_candidates:
    rankings = [fuse(item['branches'], base['k'], base['weights']) for item in qa_cache]
    # Precompute once for the largest pool; smaller pools reuse pair scores.
    for i, (item, ranked) in enumerate(zip(qa_cache, rankings, strict=True)):
        with capture_output():
            qa_reranker.score(item['case']['query'], [r['embedding_text'] for r in ranked[:30]])
        if (i + 1) % 10 == 0:
            print('Cross-encoder train', base['configuration'], i + 1, '/200', flush=True)
    for pool in (10, 20, 30):
        for weight in (.25, .5, .75, 1.0):
            rows = []
            for item, ranked in zip(qa_cache, rankings, strict=True):
                reranked = rerank_evidence(item['case']['query'], ranked, qa_reranker,
                    pool, weight, base['k'])
                rows.append(qa_direct_coverage(item['case'], reranked[:10]))
            means = pd.DataFrame(rows)[['all_groups@5', 'all_groups@10',
                'group_recall@10', 'mrr@10']].mean().to_dict()
            qa_advanced.append({**base, **means, 'configuration': 'C11',
                'base_configuration': base['configuration'], 'reranker': 'cross_encoder',
                'rerank_pool': pool, 'rank_weight': weight})
    print('Cross-encoder policy comparison complete.', flush=True)
display(pd.DataFrame(qa_advanced))
qa_winner = max([qa_base, *qa_advanced], key=qa_objective)
qa_policy = {**qa_winner, 'pool': 100,
    'selection_status': 'SUPPLIED_200_TRAIN_ONLY_STRICT_PRIMARY_CHUNK_IDS',
    'objective': ['all_groups@10', 'group_recall@10', 'all_groups@5', 'mrr@10'],
    'train_sha256': digest(qa_train), 'train_count': 200,
    'base_trials': len(qa_trials), 'advanced_trials': len(qa_advanced),
    'candidate_types': QA_CHUNK_TYPES,
    'limitations': 'Best in tested grid; no model fine-tuning; C12 not retrained in this QA experiment.'}
qa_frozen_selection = {**hs_before_qa, 'qa': qa_policy}
assert {k:v for k,v in qa_frozen_selection.items() if k != 'qa'} == {
    k:v for k,v in hs_before_qa.items() if k != 'qa'}
write_json(ROOT / 'config/selected_retrieval.json', qa_frozen_selection)
qa_frozen_hash = digest(qa_frozen_selection)
print('FROZEN QA POLICY before evaluating test:', qa_policy, flush=True)
print('HS policies unchanged. Selected hash:', qa_frozen_hash)
qa_conn.close()


Cached branches 1/200


Cached branches 2/200


Cached branches 3/200


Cached branches 4/200


Cached branches 5/200


...

Cross-encoder train C10 200 /200


Cross-encoder policy comparison complete.


,configuration,weights,k,all_groups@5,all_groups@10,group_recall@10,mrr@10,base_configuration,reranker,rerank_pool,rank_weight
0,C11,"{'dense': 0.35, 'sparse': 0.15, 'bm25': 0.45, ...",5,0.840,0.900,0.917667,0.737532,C13,cross_encoder,10,0.25
1,C11,"{'dense': 0.35, 'sparse': 0.15, 'bm25': 0.45, ...",5,0.860,0.900,0.917667,0.781101,C13,cross_encoder,10,0.50
2,C11,"{'dense': 0.35, 'sparse': 0.15, 'bm25': 0.45, ...",5,0.860,0.900,0.917667,0.796954,C13,cross_encoder,10,0.75
3,C11,"{'dense': 0.35, 'sparse': 0.15, 'bm25': 0.45, ...",5,0.860,0.900,0.917667,0.800083,C13,cross_encoder,10,1.00
4,C11,"{'dense': 0.35, 'sparse': 0.15, 'bm25': 0.45, ...",5,0.855,0.900,0.917667,0.743645,C13,cross_encoder,20,0.25
5,C11,"{'dense': 0.35, 'sparse': 0.15, 'bm25': 0.45, ...",5,0.870,0.910,0.926833,0.782248,C13,cross_encoder,20,0.50
6,C11,"{'dense': 0.35, 'sparse': 0.15, 'bm25': 0.45, ...",5,0.850,0.925,0.934333,0.795718,C13,cross_encoder,20,0.75
7,C11,"{'dense': 0.35, 'sparse': 0.15, 'bm25': 0.45, ...",5,0.850,0.925,0.934333,0.799200,C13,cross_encoder,20,1.00
8,C11,"{'dense': 0.35, 'sparse': 0.15, 'bm25': 0.45, ...",5,0.860,0.900,0.917667,0.747909,C13,cross_encoder,30,0.25
9,C11,"{'dense': 0.35, 'sparse': 0.15, 'bm25': 0.45, ...",5,0.865,0.920,0.936000,0.786518,C13,cross_encoder,30,0.50


FROZEN QA POLICY before evaluating test: {'configuration': 'C11', 'weights': {'dense': 0.3, 'sparse': 0.05, 'bm25': 0.3, 'fts': 0.05, 'exact': 0.3}, 'k': 5, 'all_groups@5': 0.86, 'all_groups@10': 0.925, 'group_recall@10': 0.9359999999999999, 'mrr@10': 0.7993769841269841, 'base_configuration': 'C10', 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.75, 'pool': 100, 'selection_status': 'SUPPLIED_200_TRAIN_ONLY_STRICT_PRIMARY_CHUNK_IDS', 'objective': ['all_groups@10', 'group_recall@10', 'all_groups@5', 'mrr@10'], 'train_sha256': 'e21f97a0e55eedc143471c1d2111d76afd2fd6789c777600ed633ef1fac48639', 'train_count': 200, 'base_trials': 44716, 'advanced_trials': 24, 'candidate_types': ['gir_rule', 'gir_explanation', 'section_note', 'chapter_note', 'subheading_note', 'tariff_entry', 'legal_clause'], 'limitations': 'Best in tested grid; no model fine-tuning; C12 not retrained in this QA experiment.'}


HS policies unchanged. Selected hash: 95230f6b9f14b8abfaa615294668ed86e0e32cebcb55a62b1b625e09a5909e04


### Phạm vi và giới hạn
QA tìm trên GIR, Notes, TT85 và các tầng dòng biểu thuế; không hard-filter Section sớm. Cấu hình HS vẫn được giữ nguyên. Top-10 QA và gói căn cứ bổ sung được đánh giá riêng. Có 8 câu test chung căn cứ nguồn với train; không có câu hỏi trùng. C12 có chi phí cao và chưa được tối ưu lại trong vòng QA này.

**Chẩn đoán vòng QA đã chạy:** C11 đạt đủ primary Top-10 185/200 trên train, nhưng chỉ 26/50 trên test; gói trực tiếp + graph lúc đó đạt 27/50. Train/test khác phân bố mạnh. Nhiều lỗi test là lấy đúng HS6/AHTN8 nhưng chưa lấy heading nguồn yêu cầu. Đã giữ nguyên cấu hình, bổ sung tài liệu cha thật và Notes vào gói căn cứ; Top-10 được giữ nguyên để không đánh đồng bổ sung bằng chứng với cải thiện thứ hạng.

## QA: kết quả lịch sử 50 test

Để xem so sánh QA mới, chạy phần QA của notebook 01 trên snapshot active sau notebook 01. Output ở đây được giữ nguyên làm mốc lịch sử.


In [3]:
# Test only after train selection/freeze; only query strings are passed to runtime.
import ast
import io

import nbformat
import numpy as np
import pandas as pd
from IPython.display import display
from IPython.utils.io import capture_output

from rag.data import ROOT, config, digest, load_corpus, read
from rag.db import connect
from rag.embedding import Embedder
from rag.evaluation import discover_gold, discover_qa, qa_direct_coverage, query_for
from rag.retrieval import Retriever

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

qa_frozen_hash = digest(read(ROOT / 'config/selected_retrieval.json'))
qa_all_current = discover_qa(load_corpus())
qa_test_current = [c for c in qa_all_current if c['split'] == 'test']
qa_test_rows = []
with connect() as qa_test_conn:
    with capture_output():
        qa_test_embedder = Embedder(config())
    qa_test_runtime = Retriever(qa_test_conn, qa_test_embedder)
    for case in qa_test_current:
        with capture_output():
            result = qa_test_runtime.qa_search(case['query'], top_k=10)
        strict = qa_direct_coverage(case, result['matches'])
        required = {g['any_of'][0] for g in case['canonical_groups']}
        direct_package = {r['id'] for r in result['package']}
        strict.update({'category': case['category'], 'elapsed_ms': result['elapsed_ms'],
            'package_primary_complete': required <= direct_package,
            'package_primary_recall': len(required & direct_package) / len(required),
            'package_size': len(result['package']),
            'missing_primary_ids': sorted(required - direct_package),
            'gold_overlap_with_train': case['has_cross_subset_exact_gold_overlap'],
            'ancestor_ids': [r['id'] for r in result['ancestors']['records']],
            'notes_ids': [r['id'] for r in result['notes']],
            'code_mentions': result['query_plan']['code_mentions']})
        qa_test_rows.append(strict)
        print('TEST', case['id'], 'primary recall@10', strict['group_recall@10'], flush=True)
    # Diagnostic regression against saved HS test outputs; QA config must not alter HS proposals.
    saved = nbformat.read(ROOT / 'notebooks/02_test_retrieval.ipynb', as_version=4)
    hs_tables = [pd.read_html(io.StringIO(o.data['text/html']))[0]
        for o in saved.cells[29].outputs
        if o.output_type in ('display_data', 'execute_result') and 'text/html' in o.data]
    hs_old = next(t for t in hs_tables if 'after_top5' in t)
    hs_gold, _ = discover_gold()
    hs_products = {r['test_id']:r for r in hs_gold if r['split']=='test'}
    hs_qa_regression = []
    for _, old in hs_old.iterrows():
        base_id, variant = old['test_id'].split(':')
        with capture_output():
            actual = qa_test_runtime.routed_search(query_for(hs_products[base_id], variant))
        codes = [r['code'] for r in actual.get('candidates',[])[:5]]
        expected = [str(c) for c in ast.literal_eval(old['after_top5'])]
        hs_qa_regression.append({'test_id': old['test_id'], 'top5_unchanged': codes == expected})
    assert len(hs_qa_regression) == 180 and all(r['top5_unchanged'] for r in hs_qa_regression)
assert digest(read(ROOT / 'config/selected_retrieval.json')) == qa_frozen_hash
qa_frame = pd.DataFrame(qa_test_rows).drop(columns=[
    'package_all_groups', 'package_group_recall', 'missing_package_groups'], errors='ignore')
columns = ['all_groups@5', 'all_groups@10', 'group_recall@10', 'mrr@10',
    'package_primary_complete', 'package_primary_recall', 'citation_complete']
display(pd.DataFrame([{'count': len(qa_frame), **qa_frame[columns].mean().to_dict(),
    'latency_p50_ms': qa_frame.elapsed_ms.quantile(.5), 'latency_p95_ms': qa_frame.elapsed_ms.quantile(.95),
    'snapshot': qa_test_runtime.snapshot, 'policy_hash': qa_frozen_hash}]))
display(qa_frame.groupby('category')[columns].mean().join(qa_frame.groupby('category').size().rename('count')))
display(qa_frame.groupby('gold_overlap_with_train')[columns].mean().join(
    qa_frame.groupby('gold_overlap_with_train').size().rename('count')))
display(qa_frame)
display(qa_frame.loc[~qa_frame.package_primary_complete,
    ['case','query','missing_primary_ids','returned_top10']])
print('HS regression:', sum(r['top5_unchanged'] for r in hs_qa_regression), '/180 unchanged.')
print('50 supplied test queries; 8 share gold source chunks with train. Test is an existing diagnostic set, not independent fresh proof.')
print('Train selects retrieval weights only; no gold ingestion, model fine-tuning, or automatic final-HS/legal decision.')


TEST LQ001 primary recall@10 0.0


TEST LQ002 primary recall@10 0.0


TEST LQ003 primary recall@10 0.0


TEST LQ004 primary recall@10 0.0


TEST LQ005 primary recall@10 1.0


...

,all_groups@5,all_groups@10,group_recall@10,mrr@10,package_primary_complete,package_primary_recall,citation_complete,count
category,,,,,,,,
cross_section_exclusion,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,1.0,2
gir_rule_retrieval,0.750000,0.750000,0.916667,0.800000,1.000000,1.000000,1.0,4
heading_or_subheading_legal_retrieval,0.448276,0.482759,0.482759,0.247126,0.896552,0.896552,1.0,29
heading_plus_legal_note,0.363636,0.454545,0.500000,0.232684,1.000000,1.000000,1.0,11
legal_note_retrieval,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,2
multi_source_legal_retrieval,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,2


,all_groups@5,all_groups@10,group_recall@10,mrr@10,package_primary_complete,package_primary_recall,citation_complete,count
gold_overlap_with_train,,,,,,,,
False,0.428571,0.47619,0.503968,0.260147,0.928571,0.928571,1.0,42
True,0.750000,0.75000,0.750000,0.750000,0.750000,0.875000,1.0,8


,case,query,required_groups,group_recall@5,all_groups@5,group_recall@10,all_groups@10,package_group_recall,package_all_groups,missing_package_groups,mrr@10,group_discounted_coverage@10,returned_top10,package_size,citation_complete,category,elapsed_ms,package_primary_complete,package_primary_recall,missing_primary_ids,gold_overlap_with_train,ancestor_ids,notes_ids,code_mentions
0,LQ001,"Ngọc trai tự nhiên hoặc nuôi cấy, chưa xâu thành chuỗi, chưa gắn hoặc nạm dát thuộc nhóm nào?",1,0.000000,False,0.000000,False,0.000000,False,[tt31:HS4:7101],0.000000,0.000000,"[tt31:HS6:710121, tt31:HS6:710122, tt31:HS8:71049900, tt31:HS8:71012100, tt31:HS8:71042900, tt31:HS6:710499, tt31:HS8:71012200, tt31:HS8:71011000, tt31:HS6:710110, tt31:HS6:710491]",24,True,heading_or_subheading_legal_retrieval,1545.403458,True,1.0,[],False,"[tt31:CH:71, tt31:HS4:7101, tt31:HS4:7104, tt31:HS6:710429]","[tt31:XIV_CH71_NOTE_01, tt31:XIV_CH71_NOTE_11, tt31:XIV_CH71_NOTE_03, tt31:XIV_CH71_NOTE_09, tt31:XIV_CH71_NOTE_02, tt31:XIV_CH71_NOTE_06, tt31:XIV_CH71_NOTE_08, tt31:XIV_CH71_SUBHEADING_NOTE_03, tt31:XIV_CH71_NOTE_04, tt31:XIV_CH71_NOTE_10]",[]
1,LQ002,Kim cương phi công nghiệp đã gia công nhưng chưa gắn hoặc nạm dát thuộc nhóm nào?,1,0.000000,False,0.000000,False,0.000000,False,[tt31:HS4:7102],0.000000,0.000000,"[tt31:HS6:710231, tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS6:710239, tt31:HS8:71023900, tt31:HS8:71023100, tt31:HS6:710229, tt31:HS6:710221, tt31:HS8:71022100, tt31:HS8:71022900, tt31:HS8:71021000]",23,True,heading_or_subheading_legal_retrieval,120.854625,True,1.0,[],False,"[tt31:CH:71, tt31:HS4:7102, tt31:HS6:710210]","[tt31:XIV_CH71_NOTE_03, tt31:XIV_CH71_NOTE_11, tt31:XIV_CH71_NOTE_06, tt31:XIV_CH71_NOTE_02, tt31:XIV_CH71_NOTE_01, tt31:XIV_CH71_NOTE_04, tt31:XIV_CH71_NOTE_07, tt31:XIV_CH71_NOTE_09, tt31:XIV_CH71_SUBHEADING_NOTE_03, tt31:XIV_CH71_SUBHEADING_NOTE_02]",[]
2,LQ003,"Rubi tự nhiên, chưa gắn hoặc nạm dát, thuộc nhóm nào?",1,0.000000,False,0.000000,False,0.000000,False,[tt31:HS4:7103],0.000000,0.000000,"[tt31:HS8:71039190, tt31:HS8:71039110, tt31:HS8:71031010, tt31:HS6:710391, tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS4:7102, tt31:HS6:710210, tt31:HS6:710231, tt31:HS6:710239, tt31:HS8:71042900]",25,True,heading_or_subheading_legal_retrieval,105.515750,True,1.0,[],False,"[tt31:CH:71, tt31:HS4:7103, tt31:HS4:7104, tt31:HS6:710310, tt31:HS6:710429]","[tt31:XIV_CH71_NOTE_01, tt31:XIV_CH71_NOTE_03, tt31:XIV_CH71_NOTE_11, tt31:XIV_CH71_SUBHEADING_NOTE_03, tt31:XIV_CH71_NOTE_02, tt31:XIV_CH71_NOTE_08, tt31:XIV_CH71_SUBHEADING_NOTE_02, tt31:XIV_CH71_NOTE_09, tt31:XIV_CH71_NOTE_04, tt31:XIV_CH71_NOTE_06]",[]
3,LQ004,"Saphia tổng hợp hoặc tái tạo, chưa gắn hoặc nạm dát, thuộc nhóm nào?",1,0.000000,False,0.000000,False,0.000000,False,[tt31:HS4:7104],0.000000,0.000000,"[tt31:SXVI-N-SXVI_NOTE_1F, tt31:XIV_CH71_NOTE_03, tt31:HS6:710499, tt31:HS8:71039190, tt31:XIV_CH71_NOTE_11, tt31:HS6:710491, tt31:HS8:71039110, tt31:HS8:71049900, tt31:HS6:710429, tt31:HS6:710391]",21,True,heading_or_subheading_legal_retrieval,106.548500,True,1.0,[],False,"[tt31:CH:71, tt31:HS4:7103, tt31:HS4:7104]","[tt31:XIV_CH71_NOTE_11, tt31:XIV_CH71_NOTE_03, tt31:XIV_CH71_NOTE_01, tt31:XIV_CH71_NOTE_09, tt31:XIV_CH71_NOTE_02, tt31:XIV_CH71_SUBHEADING_NOTE_03, tt31:XIV_CH71_NOTE_06, tt31:XIV_CH71_NOTE_08, tt31:XIV_CH71_NOTE_05, tt31:XIV_CH71_NOTE_10]",[]
4,LQ005,Vàng chưa gia công thuộc phạm vi nhóm nào của Chương 71?,1,1.000000,True,1.000000,True,1.000000,True,[],0.250000,0.430677,"[tt31:HS6:710812, tt31:HS8:71081210, tt31:HS8:71081290, tt31:HS4:7108, tt31:HS6:710811, tt31:HS6:710820, tt31:HS6:710813, tt31:HS8:71069100, tt31:HS8:71081300, tt31:HS8:71081100]",23,True,heading_or_subheading_legal_retrieval,100.879584,True,1.0,[],False,"[tt31:CH:71, tt31:HS4:7106, tt31:HS6:710691]","[tt31:XIV_CH71_NOTE_03, tt31:XIV_CH71_NOTE_02, tt31:XIV_CH71_NOTE_10, tt31:XIV_CH71_NOTE_08, tt31:XIV_CH71_NOTE_11, tt31:XIV_CH71_NOTE_04, tt31:XIV_CH71_NOTE_01, tt31:XIV_CH71_NOTE_09, tt31:XIV_CH71_NOTE_05, tt31:XIV_C

,case,query,missing_primary_ids,returned_top10
16,LQ017,"Pin sơ cấp, không thể nạp lại, cần phân biệt với ắc quy nạp lại ở heading nào?",[tt31:SXVI-T-8506],"[tt31:SXVI-N-CH85_SUBNOTE_5, tt31:SXVI-T-854914, tt31:SXVI-T-854991, tt31:SXVI-T-85491119, tt31:SXVI-T-854913, tt31:SXVI-T-854919, tt31:SXVI-T-85491199, tt31:SXVI-T-85491111, tt31:SXVI-T-85491410, tt31:SXVI-T-85491191]"
22,LQ023,Mạch in chưa lắp linh kiện điện tử thuộc heading nào?,[tt31:SXVI-T-8534],"[tt31:SXVI-N-CH85_NOTE_8, tt31:SXVI-T-84798910, tt31:SXVI-T-84733010, tt31:SXVI-N-CH85_NOTE_6A, tt31:SXVI-T-84862041, tt31:SXVI-T-84869046, tt31:SXVI-T-85049020, tt31:SXVI-T-84862042, tt31:SXVI-N-SXVI_NOTE_6A_INTRO, tt31:SXVI-T-85159090]"
26,LQ027,Động cơ điện được chế tạo riêng để lắp trên ô tô có mặc nhiên là phụ tùng 87.08 không; cần đối chiếu heading nào?,[tt31:SXVI-T-8501],"[tt31:SXVII_HS8_87085099, tt31:SXVII_HS8_87085027, tt31:SXVII_HS8_87087023, tt31:SXVII_HS8_87085013, tt31:SXVII_HS8_87085025, tt31:SXVII_HS8_87085015, tt31:SXVII_HS8_87085011, tt31:SXVII_HS8_87085026, tt31:SXVI-T-85114032, tt31:SXVII_HS8_87071030]"
27,LQ028,Ổ bi hoặc ổ đũa chuyên dùng cho bánh xe ô tô có mặc nhiên là phụ tùng 87.08 không; cần đối chiếu heading nào?,[tt31:SXVI-T-8482],"[tt31:SXVII_HS8_87087023, tt31:SXVII_HS8_87087032, tt31:SXVII_HS8_87087099, tt31:SXVII_HS8_87087095, tt31:SXVII_HS8_87087031, tt31:SXVII_HS8_87087097, tt31:SXVII_HS8_87085096, tt31:SXVII_HS8_87087022, tt31:SXVII_HS8_87087015, tt31:SXVII_HS8_87087016]"
35,LQ036,Xe có động cơ dùng để chở hàng thuộc heading nào?,[tt31:SXVII_HEADING4_8704],"[tt31:SXVII_HS8_87059090, tt31:SXVII_HS8_87039019, tt31:SXVII_HS8_87039099, tt31:SXVII_HS8_87033340, tt31:SXVII_HS8_87032421, tt31:SXVII_HS8_87021050, tt31:SXVII_HS8_87071090, tt31:SXVII_HS8_87021079, tt31:SXVII_HS8_87021099, tt31:SXVII_HS8_87021089]"


HS regression: 180 /180 unchanged.
50 supplied test queries; 8 share gold source chunks with train. Test is an existing diagnostic set, not independent fresh proof.
Train selects retrieval weights only; no gold ingestion, model fine-tuning, or automatic final-HS/legal decision.


## New verified HS partitions — active snapshot

Train selects candidate-generation and fusion policies. Test is diagnostic because earlier tests were inspected. Holdout labels are loaded **only after** the selection file has been frozen. Outputs stay in this notebook; branch caches are runtime artifacts.

**Chạy workflow hiện hành:** chọn các cell từ phần này đến cuối notebook rồi Run Selected Cells. Các cell phía trên lưu thí nghiệm/snapshot lịch sử và output cũ; chúng không thuộc lần chọn cấu hình 709/143/88.


In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
candidates = [Path(os.environ["AIAGENT_ROOT"])] if "AIAGENT_ROOT" in os.environ else []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "apps/aiAgent", parent / "frappe-bench/apps/aiAgent"])
ROOT = next((path.resolve() for path in candidates if (path / "rag/data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set AIAGENT_ROOT to the absolute aiAgent folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from collections import Counter  # noqa: E402

import pandas as pd  # noqa: E402
from IPython.display import display  # noqa: E402

from rag.data import ROOT, config, digest, load_corpus, read  # noqa: E402
from rag.db import connect  # noqa: E402
from rag.embedding import Embedder  # noqa: E402
from rag.evaluation import benchmark_split, branch_cache, discover_gold, query_for  # noqa: E402
from rag.retrieval import Retriever  # noqa: E402

cfg = config()
bundle = load_corpus()
conn = connect()
embedder = Embedder(cfg)
retriever = Retriever(conn, embedder)
assert retriever.snapshot == bundle["snapshot"]
train_test, source_files = discover_gold(("train", "test"))
assert {r["expected_hs_8"] for r in train_test} <= {r.get("code") for r in bundle["records"]}
partition_audit = benchmark_split(train_test)
train_new = [{**r, "query": query_for(r, "name_and_spec")} for r in train_test if r["split"] == "train"]
test_new = [{**r, "query": query_for(r, "name_and_spec")} for r in train_test if r["split"] == "test"]
print("Active snapshot", retriever.snapshot, "profile", retriever.profile)
print("Train/test sources", source_files, "counts", len(train_new), len(test_new))
print("Train/test hash", digest(train_test))
display(pd.DataFrame(train_test).groupby(["split", "section"]).agg(cases=("test_id", "count"), hs8=("expected_hs_8", "nunique")))

/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Fetching 22 files: 100%|██████████| 22/22 [00:00<00:00, 4962.87it/s]

Active snapshot e2354dac4a5bdc2c717cac99 profile 3a2ce6e6d726a0f053748717
Train/test sources ['data/verified/hs/train/HS_RAG_TRAIN.jsonl', 'data/verified/hs/test/HS_RAG_TEST.jsonl'] counts 709 143
Train/test hash 669cc5122a218044a23041120bee544d57e68028e555a7389f96d1a0966287ce


cases  hs8
split section            
test  XIV         48   14
      XVI         47   47
      XVII        48   47
train XIV        167   80
      XVI        350  325
      XVII       192  158

In [2]:
from IPython.utils.io import capture_output

with capture_output() as progress:
    train_leaf_cache = branch_cache(retriever, train_new, filters={"levels": [8]}, pool=100)
print("Cached TRAIN leaf branches:", len(train_leaf_cache))
print("First query timing (ms):", train_leaf_cache[0]["timings"])


Cached TRAIN leaf branches: 709
First query timing (ms): {'embedding_ms': 1.6464170003018808, 'query_cache_hit': True, 'dense_ms': 29.792332999932114, 'sparse_ms': 13.876584000172443, 'bm25_ms': 406.5897500004212, 'fts_ms': 12.980999999854248, 'exact_ms': 0.3092089991696412}


In [3]:
with capture_output() as progress:
    train_router_cache = branch_cache(
        retriever, train_new, filters={"sections": ["XIV", "XVI", "XVII"]}, pool=100
    )
print("Cached TRAIN cross-section branches:", len(train_router_cache))


Cached TRAIN cross-section branches: 709


### Grouped TRAIN selection

A deterministic 80/20 family split inside TRAIN selects weights on the validation fifth. The supplied TEST and HOLDOUT do not choose weights. Compare C1–C15, several RRF k values, and a source-aware parent branch.

In [4]:
import hashlib

from rag.evaluation import CONFIGURATIONS, ranking_metrics, sweep_router, sweep_weights
from rag.retrieval import fuse, unique_codes


def validation_family(case):
    value = case.get("case_family_id") or case["test_id"]
    return int(hashlib.sha256(str(value).encode()).hexdigest(), 16) % 5 == 0


def measure_fusion(items, policy):
    rankings = [unique_codes(fuse(
        {branch: item["branches"][branch] for branch in policy["weights"]},
        policy["k"], policy["weights"]), 10) for item in items]
    return ranking_metrics([item["case"] for item in items], rankings)

family_trials, selected_base = [], {}
for section in ("XIV", "XVI", "XVII"):
    all_items = [item for item in train_leaf_cache if item["case"]["section"] == section]
    tune = [item for item in all_items if not validation_family(item["case"])]
    validation = [item for item in all_items if validation_family(item["case"])]
    assert len(tune) > 20 and len(validation) > 10
    shortlist = []
    for family in CONFIGURATIONS:
        for row in sweep_weights(tune, family, (5, 20, 60, 100))[:5]:
            metric = measure_fusion(validation, row)
            shortlist.append({"section": section, "configuration": family,
                "weights": row["weights"], "k": row["k"],
                "tune_hit@1": row["hit@1"], "validation_hit@1": metric["hit@1"],
                "validation_recall@5": metric["ahtn8_recall@5"],
                "validation_mrr": metric["mrr"], "validation_count": len(validation)})
        print("Swept", section, family, flush=True)
    old_policy = read(ROOT / "config/selected_retrieval.json")["section_policies"][section]
    old_metric = measure_fusion(validation, old_policy)
    shortlist.append({"section": section, "configuration": old_policy["configuration"],
        "weights": old_policy["weights"], "k": old_policy["k"],
        "tune_hit@1": None, "validation_hit@1": old_metric["hit@1"],
        "validation_recall@5": old_metric["ahtn8_recall@5"],
        "validation_mrr": old_metric["mrr"], "validation_count": len(validation),
        "source": "previous_policy"})
    family_trials.extend(shortlist)
    chosen = max(shortlist, key=lambda r: (
        r["validation_hit@1"], r["validation_recall@5"],
        r["validation_mrr"], -len(r["weights"]), -r["k"]))
    selected_base[section] = {key: chosen[key] for key in ("configuration", "weights", "k")}
    print("Selected", section, selected_base[section], flush=True)
print("Family trials:", len(family_trials))
display(pd.DataFrame(family_trials).sort_values(
    ["section", "validation_hit@1", "validation_recall@5"],
    ascending=[True, False, False]).groupby("section").head(10))


Swept XIV C1


Swept XIV C2


Swept XIV C3


Swept XIV C4


Swept XIV C5


...

Swept XVII C14


Swept XVII C15


Selected XVII {'configuration': 'C13', 'weights': {'dense': 0.00012596190210462563, 'sparse': 0.3149708899853406, 'bm25': 0.6848674538035058, 'exact': 3.56943090491384e-05}, 'k': 60}


Family trials: 186


,section,configuration,weights,k,tune_hit@1,validation_hit@1,validation_recall@5,validation_mrr,validation_count,source
61,XIV,C9,"{'dense': 0.35, 'bm25': 0.35, 'exact': 0.3}",20,NaN,0.205882,0.382353,0.286146,34,previous_policy
0,XIV,C1,{'dense': 1.0},5,0.187970,0.176471,0.411765,0.280742,34,NaN
1,XIV,C1,{'dense': 1.0},20,0.187970,0.176471,0.411765,0.280742,34,NaN
2,XIV,C1,{'dense': 1.0},60,0.187970,0.176471,0.411765,0.280742,34,NaN
3,XIV,C1,{'dense': 1.0},100,0.187970,0.176471,0.411765,0.280742,34,NaN
8,XIV,C3,{'bm25': 1.0},5,0.225564,0.176471,0.411765,0.251961,34,NaN
9,XIV,C3,{'bm25': 1.0},20,0.225564,0.176471,0.411765,0.251961,34,NaN
10,XIV,C3,{'bm25': 1.0},60,0.225564,0.176471,0.411765,0.251961,34,NaN
11,XIV,C3,{'bm25': 1.0},100,0.225564,0.176471,0.411765,0.251961,34,NaN
21,XIV,C6,"{'dense': 0.3074606737585251, 'bm25': 0.692539...",5,0.233083,0.176471,0.411765,0.259921,34,NaN


In [5]:
from rag.evaluation import sweep_router
from rag.retrieval import fuse


def validation_family(case):
    value = case.get("case_family_id") or case["test_id"]
    return int(hashlib.sha256(str(value).encode()).hexdigest(), 16) % 5 == 0

router_tune = [item for item in train_router_cache if not validation_family(item["case"])]
router_validation = [item for item in train_router_cache if validation_family(item["case"])]
router_shortlist = sweep_router(router_tune)[:30]


def router_metrics(items, policy):
    by_section = {}
    for section in ("XIV", "XVI", "XVII"):
        subset = [item for item in items if item["case"]["section"] == section]
        predictions = [retriever.section_candidates(fuse(
            item["branches"], policy["k"], policy["weights"])) for item in subset]
        by_section[section] = {
            "accuracy": sum(bool(p) and p[0]["section"] == section for p in predictions) / len(subset),
            "recall@2": sum(any(x["section"] == section for x in p[:2]) for p in predictions) / len(subset),
        }
    return {
        "accuracy_macro": sum(x["accuracy"] for x in by_section.values()) / 3,
        "recall@2_macro": sum(x["recall@2"] for x in by_section.values()) / 3,
        "by_section": by_section,
    }

router_trials = [{**policy, **router_metrics(router_validation, policy)}
                 for policy in router_shortlist]
old_selection = read(ROOT / "config/selected_retrieval.json")
old_router = {key: old_selection["router"][key] for key in ("weights", "k")}
old_router_validation = router_metrics(router_validation, old_router)
router_trials.append({**old_router, **old_router_validation,
    "section_accuracy_macro": None, "section_recall@2_macro": None,
    "source": "previous_policy"})
selected_router_trial = max(router_trials, key=lambda r: (
    r["recall@2_macro"], r["accuracy_macro"], -r["k"]))
selected_router = {key: selected_router_trial[key] for key in ("weights", "k")}
print("Selected router", selected_router, router_metrics(train_router_cache, selected_router))
print("Old router validation", old_router_validation)
display(pd.DataFrame([{key: row[key] for key in (
    "k", "section_accuracy_macro", "section_recall@2_macro", "accuracy_macro", "recall@2_macro")}
    for row in router_trials[:10]]))


Selected router {'weights': {'dense': 0.7, 'sparse': 0.1, 'bm25': 0.1, 'fts': 0.05, 'exact': 0.05}, 'k': 60} {'accuracy_macro': 0.9928869047619048, 'recall@2_macro': 1.0, 'by_section': {'XIV': {'accuracy': 1.0, 'recall@2': 1.0}, 'XVI': {'accuracy': 0.9942857142857143, 'recall@2': 1.0}, 'XVII': {'accuracy': 0.984375, 'recall@2': 1.0}}}
Old router validation {'accuracy_macro': 0.9831746031746031, 'recall@2_macro': 1.0, 'by_section': {'XIV': {'accuracy': 1.0, 'recall@2': 1.0}, 'XVI': {'accuracy': 0.9733333333333334, 'recall@2': 1.0}, 'XVII': {'accuracy': 0.9761904761904762, 'recall@2': 1.0}}}


,k,section_accuracy_macro,section_recall@2_macro,accuracy_macro,recall@2_macro
0,60,0.995556,1.0,0.983175,1.0
1,60,0.993333,1.0,0.983175,1.0
2,60,0.993333,1.0,0.983175,1.0
3,100,0.993333,1.0,0.983175,1.0
4,100,0.993333,1.0,0.983175,1.0
5,60,0.993049,1.0,0.983175,1.0
6,60,0.992121,1.0,0.983175,1.0
7,60,0.992121,1.0,0.983175,1.0
8,100,0.992121,1.0,0.983175,1.0
9,60,0.991837,1.0,0.983175,1.0


In [6]:
import gzip
import json

from rag.data import digest
from rag.retrieval import candidate_paths, definition_fusion

router_by_id = {item["case"]["test_id"]: item for item in train_router_cache}
base_params = {"strategy": "direct", "candidate_pool": 100,
               "section_mix": "router", "beam_size": 10, "k": 20,
               "reranker": "none"}
parent_cache_dir = ROOT / "artifacts/benchmark_parents"
parent_cache_dir.mkdir(parents=True, exist_ok=True)
train_pools = []
for index, item in enumerate(train_leaf_cache, 1):
    case = item["case"]
    router_item = router_by_id[case["test_id"]]
    sections = retriever.section_candidates(fuse(
        router_item["branches"], selected_router["k"], selected_router["weights"]))
    assert sections
    direct = candidate_paths(item["branches"], sections, selected_base, base_params)["candidates"]
    section_names = [s["section"] for s in sections]
    key = digest({"snapshot": retriever.snapshot, "query": case["query"],
                  "sections": section_names, "parent_limit": 20, "child_limit": 5})
    path = parent_cache_dir / (key + ".json.gz")
    if path.exists():
        with gzip.open(path, "rt") as stream:
            parents = json.load(stream)
    else:
        dense_vector = embedder.encode([case["query"]])["dense"][0]
        parents = retriever.parent_candidates(case["query"], dense_vector, section_names)
        with gzip.open(path, "wt") as stream:
            json.dump(parents, stream, ensure_ascii=False, default=str)
    definitions = retriever.definition_candidates(case["query"], 100, section_names)
    train_pools.append({"case": case, "sections": sections, "direct": direct,
                        "parents": parents, "definitions": definitions})
    if index % 100 == 0:
        print("Parent/definition pools", index, "/", len(train_leaf_cache), flush=True)
print("Prepared", len(train_pools), "train candidate pools")


Parent/definition pools 100 / 709


Parent/definition pools 200 / 709


Parent/definition pools 300 / 709


Parent/definition pools 400 / 709


Parent/definition pools 500 / 709


Parent/definition pools 600 / 709


Parent/definition pools 700 / 709


Prepared 709 train candidate pools


In [7]:
from rag.evaluation import ranking_metrics


def pool_ranking(item, parent_weight, definition_weight, k=60):
    rows = definition_fusion(item["direct"], item["parents"], parent_weight, k)
    return definition_fusion(rows, item["definitions"], definition_weight, k)

branch_trials, selected_support = [], {}
for routed_section in ("XIV", "XVI", "XVII"):
    section_items = [item for item in train_pools
                     if item["sections"][0]["section"] == routed_section]
    tune = [item for item in section_items if not validation_family(item["case"])]
    validation = [item for item in section_items if validation_family(item["case"])]
    assert tune and validation
    baseline = ranking_metrics([x["case"] for x in validation],
        [pool_ranking(x, 0, 0) for x in validation])
    for parent_weight in (0, 0.1, 0.2, 0.35, 0.5):
        for definition_weight in (0, 0.2, 0.5):
            tune_score = ranking_metrics([x["case"] for x in tune],
                [pool_ranking(x, parent_weight, definition_weight) for x in tune])
            val_score = ranking_metrics([x["case"] for x in validation],
                [pool_ranking(x, parent_weight, definition_weight) for x in validation])
            branch_trials.append({"section": routed_section,
                "parent_weight": parent_weight, "definition_weight": definition_weight,
                "tune_hit@1": tune_score["hit@1"],
                "validation_hit@1": val_score["hit@1"],
                "validation_recall@5": val_score["ahtn8_recall@5"],
                "validation_mrr": val_score["mrr"],
                "validation_count": len(validation)})
    options = [x for x in branch_trials if x["section"] == routed_section
               and x["validation_recall@5"] >= baseline["ahtn8_recall@5"]]
    best = max(options, key=lambda x: (x["validation_hit@1"],
        x["validation_recall@5"], x["validation_mrr"],
        -(x["parent_weight"] + x["definition_weight"])))
    selected_support[routed_section] = best
    print("Support", routed_section, "baseline", baseline["hit@1"],
          baseline["ahtn8_recall@5"], "selected", best, flush=True)
display(pd.DataFrame(branch_trials).sort_values(
    ["section", "validation_hit@1", "validation_recall@5"],
    ascending=[True, False, False]).groupby("section").head(8))


Support XIV baseline 0.20588235294117646 0.38235294117647056 selected {'section': 'XIV', 'parent_weight': 0.5, 'definition_weight': 0, 'tune_hit@1': 0.2631578947368421, 'validation_hit@1': 0.3235294117647059, 'validation_recall@5': 0.6176470588235294, 'validation_mrr': 0.4235294117647059, 'validation_count': 34}


Support XVI baseline 0.8513513513513513 0.918918918918919 selected {'section': 'XVI', 'parent_weight': 0, 'definition_weight': 0, 'tune_hit@1': 0.8303249097472925, 'validation_hit@1': 0.8513513513513513, 'validation_recall@5': 0.918918918918919, 'validation_mrr': 0.887065637065637, 'validation_count': 74}


Support XVII baseline 0.37209302325581395 0.627906976744186 selected {'section': 'XVII', 'parent_weight': 0.1, 'definition_weight': 0, 'tune_hit@1': 0.47297297297297297, 'validation_hit@1': 0.3953488372093023, 'validation_recall@5': 0.6976744186046512, 'validation_mrr': 0.5021963824289407, 'validation_count': 43}


,section,parent_weight,definition_weight,tune_hit@1,validation_hit@1,validation_recall@5,validation_mrr,validation_count
12,XIV,0.50,0.0,0.263158,0.323529,0.617647,0.423529,34
9,XIV,0.35,0.0,0.255639,0.264706,0.588235,0.378431,34
6,XIV,0.20,0.0,0.218045,0.264706,0.529412,0.356443,34
3,XIV,0.10,0.0,0.233083,0.235294,0.529412,0.328186,34
0,XIV,0.00,0.0,0.203008,0.205882,0.382353,0.287663,34
13,XIV,0.50,0.2,0.105263,0.147059,0.529412,0.288340,34
7,XIV,0.20,0.2,0.082707,0.147059,0.470588,0.252801,34
10,XIV,0.35,0.2,0.090226,0.147059,0.441176,0.265266,34
15,XVI,0.00,0.0,0.830325,0.851351,0.918919,0.887066,74
16,XVI,0.00,0.2,0.805054,0.810811,0.918919,0.858687,74


### Source-aware Cross-Encoder on candidate pools

The parent branch does not improve rank by itself on grouped TRAIN validation. It can still help if a source-aware reranker sees its candidates. Compare direct, parent, definition, and joint pools at the same budget. Keep the direct path in every variant.

In [8]:
import torch

from rag.embedding import CrossEncoder
from rag.retrieval import rerank_candidates

reranker = CrossEncoder(cfg["reranker_model"], cfg["reranker_revision"],
    device="mps" if torch.backends.mps.is_available() else "cpu")
rerank_variants = [
    {"name": "direct", "parent_weight": 0, "definition_weight": 0, "context": False},
    {"name": "direct_context", "parent_weight": 0, "definition_weight": 0, "context": True},
    {"name": "parent_context", "parent_weight": 0.2, "definition_weight": 0, "context": True},
    {"name": "definition_context", "parent_weight": 0, "definition_weight": 0.2, "context": True},
    {"name": "joint_context", "parent_weight": 0.2, "definition_weight": 0.2, "context": True},
]
rerank_trials = []
for routed_section in ("XIV", "XVI", "XVII"):
    validation = [item for item in train_pools
                  if item["sections"][0]["section"] == routed_section
                  and validation_family(item["case"])]
    cases = [item["case"] for item in validation]
    baseline = ranking_metrics(cases, [item["direct"] for item in validation])
    rerank_trials.append({"section": routed_section, "name": "none",
        "parent_weight": 0, "definition_weight": 0, "context": False,
        "rank_weight": 0, "rerank_pool": 0,
        "hit@1": baseline["hit@1"], "recall@5": baseline["ahtn8_recall@5"],
        "mrr": baseline["mrr"], "count": len(validation)})
    for variant in rerank_variants:
        candidates = [pool_ranking(item, variant["parent_weight"],
                      variant["definition_weight"]) for item in validation]
        if variant["context"]:
            candidates = [retriever.definition_context(rows) for rows in candidates]
        for rank_weight in (0.25, 0.5, 0.75, 1.0):
            with capture_output() as rerank_progress:
                ranked = [rerank_candidates(item["case"]["query"], rows,
                    "cross_encoder", embedder, reranker, 30, rank_weight, 20)
                    for item, rows in zip(validation, candidates, strict=True)]
            metric = ranking_metrics(cases, ranked)
            rerank_trials.append({"section": routed_section, **variant,
                "rank_weight": rank_weight, "rerank_pool": 30,
                "hit@1": metric["hit@1"], "recall@5": metric["ahtn8_recall@5"],
                "mrr": metric["mrr"], "count": len(validation)})
        print("Reranked", routed_section, variant["name"], flush=True)

def rerank_objective(row):
    return (row["hit@1"], row["recall@5"], row["mrr"],
        -row["parent_weight"] - row["definition_weight"],
        row["name"] == "none")

selected_rerank = {section: max(
    (row for row in rerank_trials if row["section"] == section),
    key=rerank_objective) for section in ("XIV", "XVI", "XVII")}
for section, policy in selected_rerank.items():
    print("Selected rerank", section, policy)
display(pd.DataFrame(rerank_trials).sort_values(
    ["section", "hit@1", "recall@5", "mrr"],
    ascending=[True, False, False, False]).groupby("section").head(10))


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 2469.25it/s]

Reranked XIV direct


Reranked XIV direct_context


Reranked XIV parent_context


...

Reranked XVII parent_context


Reranked XVII definition_context


Reranked XVII joint_context


Selected rerank XIV {'section': 'XIV', 'name': 'parent_context', 'parent_weight': 0.2, 'definition_weight': 0, 'context': True, 'rank_weight': 0.5, 'rerank_pool': 30, 'hit@1': 0.29411764705882354, 'recall@5': 0.6176470588235294, 'mrr': 0.41274509803921566, 'count': 34}
Selected rerank XVI {'section': 'XVI', 'name': 'direct', 'parent_weight': 0, 'definition_weight': 0, 'context': False, 'rank_weight': 1.0, 'rerank_pool': 30, 'hit@1': 0.8783783783783784, 'recall@5': 0.9459459459459459, 'mrr': 0.9121621621621622, 'count': 74}
Selected rerank XVII {'section': 'XVII', 'name': 'parent_context', 'parent_weight': 0.2, 'definition_weight': 0, 'context': True, 'rank_weight': 0.25, 'rerank_pool': 30, 'hit@1': 0.3953488372093023, 'recall@5': 0.7209302325581395, 'mrr': 0.5027131782945736, 'count': 43}


,section,name,parent_weight,definition_weight,context,rank_weight,rerank_pool,hit@1,recall@5,mrr,count
10,XIV,parent_context,0.2,0.0,True,0.50,30,0.294118,0.617647,0.412745,34
18,XIV,joint_context,0.2,0.2,True,0.50,30,0.264706,0.617647,0.394853,34
9,XIV,parent_context,0.2,0.0,True,0.25,30,0.264706,0.588235,0.377731,34
19,XIV,joint_context,0.2,0.2,True,0.75,30,0.235294,0.588235,0.389052,34
15,XIV,definition_context,0.0,0.2,True,0.75,30,0.235294,0.588235,0.377288,34
16,XIV,definition_context,0.0,0.2,True,1.00,30,0.235294,0.588235,0.369444,34
7,XIV,direct_context,0.0,0.0,True,0.75,30,0.235294,0.588235,0.368627,34
8,XIV,direct_context,0.0,0.0,True,1.00,30,0.235294,0.588235,0.366176,34
12,XIV,parent_context,0.2,0.0,True,1.00,30,0.235294,0.588235,0.366176,34
20,XIV,joint_context,0.2,0.2,True,1.00,30,0.235294,0.588235,0.366176,34


### Freeze the selected HS policy before TEST/HOLDOUT

Only grouped TRAIN validation chose the settings below. The new parent branch remains available for research but is disabled because it did not improve the measured validation metrics. Keep the QA policy separate and unchanged.

In [9]:
from rag.data import write_json

previous = read(ROOT / "config/selected_retrieval.json")
pipeline_policies, definition_policies, parent_policies = {}, {}, {}
for section in ("XIV", "XVI", "XVII"):
    choice = selected_rerank[section]
    params = {**base_params}
    if choice["name"] != "none":
        params.update({"reranker": "cross_encoder", "rerank_pool": choice["rerank_pool"],
                       "rank_weight": choice["rank_weight"]})
    pipeline_policies[section] = params
    if choice["context"] or choice["definition_weight"] > 0:
        definition_policies[section] = {
            "definition_weight": choice["definition_weight"], "k": 60,
            "evidence_rerank": choice["context"],
            "reranker": params["reranker"],
            "rerank_pool": params.get("rerank_pool", 30),
            "rank_weight": params.get("rank_weight", 1.0),
        }
    parent_policies[section] = {"weight": choice["parent_weight"],
                                "k": 60, "parent_limit": 20, "child_limit": 5}
selection = {
    "snapshot": retriever.snapshot, "profile": retriever.profile,
    "gold_hash": digest(train_new), "retrieval": cfg,
    "selected_mode": "full_hybrid",
    "router": selected_router,
    "section_policies": selected_base,
    "pipeline_policies": pipeline_policies,
    "definition_policies": definition_policies,
    "parent_policies": parent_policies,
    "legal": previous["legal"], "qa": previous["qa"],
    "selection_scope": "GROUPED_TRAIN_80_20; TEST_DIAGNOSTIC; HOLDOUT_REGRESSION_PREVIOUSLY_EXAMINED",
    "selection_objective": "Validation Hit@1, Recall@5, MRR; parent and reranker ablations",
}
write_json(ROOT / "config/selected_retrieval.json", selection)
frozen_hash = digest(selection)
active_retriever = Retriever(conn, embedder)
assert active_retriever.snapshot == selection["snapshot"]
assert active_retriever.section_policies["XVII"]["weights"] == selected_base["XVII"]["weights"]
assert active_retriever.qa_policy == previous["qa"]
print("Frozen HS policy hash:", frozen_hash)
print("Snapshot:", selection["snapshot"], "Profile:", selection["profile"])
display(pd.DataFrame([{"section": section, "base": selected_base[section]["configuration"],
    "weights": selected_base[section]["weights"],
    "parent_weight": parent_policies[section]["weight"],
    "definition_weight": definition_policies.get(section, {}).get("definition_weight", 0),
    "reranker": pipeline_policies[section]["reranker"],
    "rerank_pool": pipeline_policies[section].get("rerank_pool", 0),
    "rank_weight": pipeline_policies[section].get("rank_weight", 0)}
    for section in ("XIV", "XVI", "XVII")]))


Frozen HS policy hash: 7901e8365eed416abf00f721a5170a201886c7a204151c8fb8bcdfcf647242ad
Snapshot: e2354dac4a5bdc2c717cac99 Profile: 3a2ce6e6d726a0f053748717


,section,base,weights,parent_weight,definition_weight,reranker,rerank_pool,rank_weight
0,XIV,C9,"{'dense': 0.35, 'bm25': 0.35, 'exact': 0.3}",0.2,0,cross_encoder,30,0.50
1,XVI,C6,"{'dense': 0.1448858592553968, 'bm25': 0.855114...",0.0,0,cross_encoder,30,1.00
2,XVII,C13,"{'dense': 0.00012596190210462563, 'sparse': 0....",0.2,0,cross_encoder,30,0.25


In [10]:
import time

import numpy as np
from IPython.utils.io import capture_output

from rag.evaluation import benchmark_split, discover_gold, query_for, ranking_metrics
from rag.retrieval import Retriever

frozen_hash = digest(read(ROOT / "config/selected_retrieval.json"))
active_retriever = Retriever(conn, embedder)
assert active_retriever.snapshot == retriever.snapshot
assert active_retriever.section_policies["XVI"]["weights"] == selected_base["XVI"]["weights"]


def run_frozen_hs(cases, partition):
    diagnosis, rankings, sections = [], [], []
    for i, case in enumerate(cases, 1):
        started = time.perf_counter()
        with capture_output():
            result = active_retriever.routed_search(case["query"])
        elapsed_ms = (time.perf_counter() - started) * 1000
        candidates = result.get("candidates", [])
        ranked_sections = result.get("section_candidates", [])
        codes = [row.get("code") for row in candidates]
        gold = case["expected_hs_8"]
        diagnosis.append({"id": case["test_id"], "section": case["section"],
            "case_type": case.get("case_type"), "query": case["query"], "gold": gold,
            "routing": result["routing"]["intent"],
            "route_status": result["routing"]["status"],
            "top2_sections": [s["section"] for s in ranked_sections[:2]],
            "top5": codes[:5], "gold_rank": codes.index(gold) + 1 if gold in codes else None,
            "latency_ms": elapsed_ms,
            "evidence_status": result.get("evidence", {}).get("coverage_status")})
        rankings.append(candidates)
        sections.append(ranked_sections)
        if i % 20 == 0:
            print(partition, i, "/", len(cases), flush=True)
    assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
    summary = []
    for section in ("XIV", "XVI", "XVII"):
        indices = [i for i, case in enumerate(cases) if case["section"] == section]
        metrics = ranking_metrics([cases[i] for i in indices],
            [rankings[i] for i in indices], [sections[i] for i in indices])
        latencies = [diagnosis[i]["latency_ms"] for i in indices]
        summary.append({"partition": partition, "section": section,
            **metrics, "latency_p50_ms": float(np.percentile(latencies, 50)),
            "latency_p95_ms": float(np.percentile(latencies, 95))})
    return diagnosis, summary

print("Frozen policy", frozen_hash, "active", active_retriever.snapshot)


Frozen policy 7901e8365eed416abf00f721a5170a201886c7a204151c8fb8bcdfcf647242ad active e2354dac4a5bdc2c717cac99


In [11]:
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
test_diagnosis, test_summary = run_frozen_hs(test_new, "TEST_DIAGNOSTIC")
display(pd.DataFrame(test_summary))
print("Route intents", pd.Series([x["routing"] for x in test_diagnosis]).value_counts().to_dict())
print("Gold missing from Top-10", [x["id"] for x in test_diagnosis if x["gold_rank"] is None])
display(pd.DataFrame(test_diagnosis))


TEST_DIAGNOSTIC 20 / 143


TEST_DIAGNOSTIC 40 / 143


TEST_DIAGNOSTIC 60 / 143


TEST_DIAGNOSTIC 80 / 143


TEST_DIAGNOSTIC 100 / 143


TEST_DIAGNOSTIC 120 / 143


TEST_DIAGNOSTIC 140 / 143


,partition,section,section_correct,section_recall@2,hit@1,mrr,ndcg@10,chapter_recall@5,heading_recall@5,hs6_recall@5,ahtn8_recall@5,count,hierarchy_metric_mode,exclusion_annotated_count,exclusion_violation_rate,latency_p50_ms,latency_p95_ms
0,TEST_DIAGNOSTIC,XIV,0.937500,0.979167,0.208333,0.271354,0.306133,0.979167,0.395833,0.375000,0.354167,48,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,139.122750,234.145023
1,TEST_DIAGNOSTIC,XVI,0.957447,1.000000,0.595745,0.726064,0.783210,1.000000,0.936170,0.914894,0.914894,47,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,136.045584,254.707834
2,TEST_DIAGNOSTIC,XVII,1.000000,1.000000,0.270833,0.468948,0.544687,1.000000,0.895833,0.833333,0.750000,48,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,195.375354,276.089245


Route intents {'product': 143}
Gold missing from Top-10 ['XIV_TE001', 'XIV_TE002', 'XIV_TE003', 'XIV_TE004', 'XIV_TE005', 'XIV_TE006', 'XIV_TE007', 'XIV_TE008', 'XIV_TE009', 'XIV_TE010', 'XIV_TE011', 'XIV_TE012', 'XIV_TE013', 'XIV_TE014', 'XIV_TE015', 'XIV_TE016', 'XIV_TE017', 'XIV_TE018', 'XIV_TE019', 'XIV_TE020', 'XIV_TE021', 'XIV_TE022', 'XVII_TE017', 'HS_HOLDOUT_XIV_006', 'HS_HOLDOUT_XIV_007', 'HOLD_XIV_031', 'HOLD_XIV_032', 'HS_HOLDOUT_XIV_042', 'HS_HOLDOUT_XVI_018', 'HOLD_XVI_041', 'HOLD_XVII_019', 'HOLD_XVII_022', 'HOLD_XVII_023', 'HOLD_XVII_026', 'HS_HOLDOUT_XVII_036', 'HOLD_XVII_043', 'HOLD_XVII_044', 'HOLD_XVII_033', 'HOLD_XVII_034', 'TRAIN_ADD_XIV_017', 'TRAIN_ADD_XVII_018']


,id,section,case_type,query,gold,routing,route_status,top2_sections,top5,gold_rank,latency_ms,evidence_status
0,XIV_TE001,XIV,legacy_test,Hoa tai bằng bạc. Hoa tai bằng bạc,71131190,product,UNCERTAIN,"[XIV, XVI]","[71110010, 71159020, 71159010, 71141100, 71070...",NaN,1951.480666,EXPANSION_COMPLETE
1,XIV_TE002,XIV,legacy_test,Dây đồng hồ bằng bạc. Dây đồng hồ bằng bạc,71131190,product,UNCERTAIN,"[XIV, XVI]","[71110010, 71159020, 71129910, 71159010, 71070...",NaN,143.964958,EXPANSION_COMPLETE
2,XIV_TE003,XIV,legacy_test,Vật trang trí đồng hồ bỏ túi (fob) bằng bạc. V...,71131190,product,UNCERTAIN,"[XIV, XVI]","[71179094, 71159020, 71110010, 71141100, 71159...",NaN,138.372916,EXPANSION_COMPLETE
3,XIV_TE004,XIV,legacy_test,Mặt dây chuyền bằng bạc. Mặt dây chuyền bằng bạc,71131190,product,UNCERTAIN,"[XIV, XVI]","[71159020, 71110010, 71129910, 71159010, 71070...",NaN,120.578833,EXPANSION_COMPLETE
4,XIV_TE005,XIV,legacy_test,Khuy măng sét bằng bạc. Khuy măng sét bằng bạc,71131190,product,UNCERTAIN,"[XIV, XVI]","[71181010, 71159020, 71189020, 71159010, 71070...",NaN,124.273750,EXPANSION_COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...
138,HOLD_XVII_033,XVII,spec_threshold,Sedan xăng 1.700 cc. Sedan chở người dùng động...,87032355,product,UNCERTAIN,"[XVII, XVI]","[87032247, 87033117, 87033147, 87032217, 87032...",NaN,275.777083,EXPANSION_COMPLETE
139,HOLD_XVII_034,XVII,spec_threshold,Ô tô 4WD xăng 2.700 cc. Ô tô chở người 4 bánh ...,87032364,product,UNCERTAIN,"[XVII, XVI]","[87033380, 87038018, 87038098, 87039018, 87039...",NaN,276.257333,EXPANSION_COMPLETE
140,XIV_TR097,XIV,legacy_verified,Dây chuyền trang sức giả bằng kim loại cơ bản....,71171920,product,UNCERTAIN,"[XIV, XVI]","[71179094, 71171920, 71179020, 71090000, 71070...",2.0,208.866041,EXPANSION_COMPLETE
141,TRAIN_ADD_XIV_017,XIV,contrast_pair,Nhẫn vàng 18K thành phẩm. Nhẫn vàng hoàn chỉnh,71131990,product,UNCERTAIN,"[XIV, XVI]","[71069200, 71159020, 71081300, 71129100, 71070...",NaN,121.013083,EXPANSION_COMPLETE


### HOLDOUT regression after the source-only corpus correction

This holdout was already examined on the previous snapshot. Use it only for regression after freezing the current policy on TRAIN. Verify the unchanged input hash, disjoint products/families, corpus code coverage, and frozen policy hash. Do not retune from these rows.

In [12]:

assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
holdout_path = ROOT / "data/verified/hs/holdout/HS_RAG_HOLDOUT.jsonl"
holdout_sha = hashlib.sha256(holdout_path.read_bytes()).hexdigest()
holdout_rows, holdout_files = discover_gold(("holdout",))
partition_audit = benchmark_split(train_test + holdout_rows)
assert holdout_rows
assert {row["expected_hs_8"] for row in holdout_rows} <= {
    row.get("code") for row in bundle["records"]}
holdout_cases = [{**row, "query": query_for(row, "name_and_spec")}
                 for row in holdout_rows]
holdout_diagnosis, holdout_summary = run_frozen_hs(holdout_cases, "HOLDOUT")
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
print("Holdout source", holdout_files, "SHA256", holdout_sha)
print("Frozen policy", frozen_hash)
display(pd.DataFrame(holdout_summary))
print("Route intents", pd.Series([x["routing"] for x in holdout_diagnosis]).value_counts().to_dict())
print("Gold missing from Top-10", [x["id"] for x in holdout_diagnosis if x["gold_rank"] is None])
display(pd.DataFrame(holdout_diagnosis))


HOLDOUT 20 / 88


HOLDOUT 40 / 88


HOLDOUT 60 / 88


HOLDOUT 80 / 88


Holdout source ['data/verified/hs/holdout/HS_RAG_HOLDOUT.jsonl'] SHA256 cd6b2c2502482a6b2917f74e23c6da2e96861eb2e47193325245773145a7c523
Frozen policy 7901e8365eed416abf00f721a5170a201886c7a204151c8fb8bcdfcf647242ad


,partition,section,section_correct,section_recall@2,hit@1,mrr,ndcg@10,chapter_recall@5,heading_recall@5,hs6_recall@5,ahtn8_recall@5,count,hierarchy_metric_mode,exclusion_annotated_count,exclusion_violation_rate,latency_p50_ms,latency_p95_ms
0,HOLDOUT,XIV,0.931034,1.000000,0.172414,0.300287,0.368581,1.000000,0.586207,0.517241,0.482759,29,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,195.265375,247.992608
1,HOLDOUT,XVI,0.896552,1.000000,0.344828,0.446552,0.489567,1.000000,0.724138,0.620690,0.586207,29,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,149.930667,270.515225
2,HOLDOUT,XVII,0.900000,0.966667,0.266667,0.353690,0.404104,0.966667,0.800000,0.666667,0.466667,30,ANCESTOR_PROJECTION_OF_LEAF_RANKING,0,None,183.266187,306.481358


Route intents {'product': 87, 'legal': 1}
Gold missing from Top-10 ['HS_HOLDOUT_XIV_021', 'HS_HOLDOUT_XIV_022', 'HOLD_XIV_029', 'HS_HOLDOUT_XIV_016', 'HS_HOLDOUT_XIV_017', 'HOLD_XIV_018', 'HOLD_XIV_008', 'HS_HOLDOUT_XIV_031', 'HS_HOLDOUT_XIV_032', 'HS_HOLDOUT_XIV_036', 'HOLD_XIV_041', 'HS_HOLDOUT_XIV_041', 'HS_HOLDOUT_XVI_006', 'HS_HOLDOUT_XVI_007', 'HS_HOLDOUT_XVI_008', 'HS_HOLDOUT_XVI_010', 'HS_HOLDOUT_XVI_011', 'HS_HOLDOUT_XVI_031', 'HS_HOLDOUT_XVI_032', 'HS_HOLDOUT_XVI_033', 'HS_HOLDOUT_XVI_034', 'HS_HOLDOUT_XVI_043', 'HS_HOLDOUT_XVI_045', 'HS_HOLDOUT_XVII_019', 'HS_HOLDOUT_XVII_021', 'HS_HOLDOUT_XVII_022', 'HS_HOLDOUT_XVII_023', 'HS_HOLDOUT_XVII_007', 'HOLD_XVII_001', 'HS_HOLDOUT_XVII_033', 'HS_HOLDOUT_XVII_041', 'HS_HOLDOUT_XVII_042', 'HS_HOLDOUT_XVII_043', 'HOLD_XVII_042', 'HOLD_XVII_031', 'HOLD_XVII_032']


,id,section,case_type,query,gold,routing,route_status,top2_sections,top5,gold_rank,latency_ms,evidence_status
0,HS_HOLDOUT_XIV_029,XIV,note_definition,Đồ trang trí bằng kim loại cơ bản dát phủ vàng...,71142000,product,UNCERTAIN,"[XIV, XVII]","[71142000, 71159020, 71141900, 71090000, 71129...",1.0,167.719125,EXPANSION_COMPLETE
1,HOLD_XIV_028,XIV,note_definition,Vòng chân dạng bangle trang sức giả bằng nhựa....,71179010,product,UNCERTAIN,"[XIV, XVII]","[71171920, 71179020, 71141900, 71171910, 71070...",6.0,211.898166,EXPANSION_COMPLETE
2,HOLD_XIV_030,XIV,note_definition,Trâm cài áo trang sức giả bằng acrylic. Trâm c...,71179020,product,UNCERTAIN,"[XIV, XVI]","[71141900, 71179020, 71171920, 71142000, 71123...",2.0,195.265375,EXPANSION_COMPLETE
3,HS_HOLDOUT_XIV_021,XIV,note_definition,Hộp đựng thuốc lá điếu bằng kim loại cơ bản dá...,71132090,product,UNCERTAIN,"[XIV, XVI]","[71159020, 71142000, 71090000, 71129100, 71110...",NaN,196.624375,EXPANSION_COMPLETE
4,HS_HOLDOUT_XIV_022,XIV,note_definition,Hộp xì-gà bằng kim loại cơ bản dát phủ bạc. Hộ...,71132090,product,UNCERTAIN,"[XIV, XVI]","[71142000, 84801000, 71159020, 71070000, 71129...",NaN,179.648959,BUDGET_LIMITED
...,...,...,...,...,...,...,...,...,...,...,...,...
83,HS_HOLDOUT_XVII_042,XVII,cross_section,Hộp số ô tô chở người đã lắp hoàn chỉnh. Hộp s...,87084026,product,UNCERTAIN,"[XVII, XVI]","[87038018, 87060033, 87038098, 87039018, 87039...",NaN,186.146208,EXPANSION_COMPLETE
84,HS_HOLDOUT_XVII_043,XVII,cross_section,Bộ ly hợp ô tô chở người. Bộ ly hợp hoàn chỉnh...,87089360,product,UNCERTAIN,"[XVII, XVI]","[87038018, 87038098, 87039018, 87039098, 87033...",NaN,201.465083,EXPANSION_COMPLETE
85,HOLD_XVII_042,XVII,cross_section,Ống xả và bộ giảm thanh cho ô tô con. Cụm ống ...,87089220,product,UNCERTAIN,"[XVII, XVI]","[87089251, 87089261, 87141030, 87141070, 87089...",NaN,176.536208,EXPANSION_COMPLETE
86,HOLD_XVII_031,XVII,spec_threshold,Xe buýt điện trên 24 tấn dạng CKD. Ô tô khách ...,87024021,product,UNCERTAIN,"[XVII, XVI]","[87113030, 87060023, 87060042, 87119060, 87114...",NaN,211.872917,EXPANSION_COMPLETE
